# Nebula V25 - answer-turn repair

Enable Kaggle **Internet**, **GPU**, and secret **NEBULA_HF_TOKEN**, then **Run all**.
Stop other trainers using this output run first. This is an experiment, not a promised upgrade.

## Why this experiment

V24 preserved some skills but gave acknowledgments in place of answers and regressed
on corrected facts. The hypothesis is that training intermediate acknowledgments
alongside final answers reinforced the wrong response pattern. This run tests a
changed repair objective and sampling mix; it cannot isolate every contributing cause.

Starts from protected **V22 final weights**, pinned at
`6c670c3670444d7cde2b17760fea1cdde4a93b18` (4,000 updates).
V24 and V23 stay preserved. Architecture remains about 100M parameters, a byte vocabulary,
and a 1,024-byte context. No pretrained model is substituted.

## Focused lessons

New original synthetic repair examples cover corrected facts, unchanged facts about
another person, switching between names/colors/locations, copying label strings,
and short multi-turn writing and practical requests.

**Only the final assistant answer and its real EOS are supervised on new repair records.**
Earlier assistant messages remain visible as context, but have no loss targets.
Intermediate acknowledgment-only turns are never selected as repair targets. This does
not remove acknowledgments from earlier V22 replay or public conversations; those retain
their original assistant-only objective.

80% of optimizer updates are assistant practice; 20% retain pinned raw-language practice.
Within assistant examples: **60% new repair / 20% V22 replay / 20% public conversations**.
Repair families favor corrections (weight 4), query switching/practical/writing (3),
ownership/multi-turn/copy (2), and field-order/location (1). Replay families are uniform.
These are update/example proportions, not byte proportions. Actual target counts are logged.

The same saved 12,000 public Smol-SmolTalk conversations are reused at a pinned revision.
Whole-conversation train/validation splits remain separate; no answers are cut to fit.
New repair values exclude the existing held-out inventory and exact benchmark prompts.
No teacher calls, private user-chat imports, broad crawls or dataset files on disk.

## Run and compare

Budget **1,000 additional updates**, peak LR **3e-6**, warmup 60 and cosine decay.
Save to HF every **100 updates** and on graceful interruption. Session cap **7.5 hours**;
preparation/evaluation/backups add time and platform GPU quotas still apply.
Baseline, update 500 and final evaluations use the existing 96 questions. Full open probes
run at baseline/final only. These questions are now a **development comparison**, because
results have informed several curricula. They are not an independent final certification.
Inspect actual replies as well as scores; familiar answers can be memorized.

A candidate must improve total exact answers, preserve corrected facts, entity binding,
unseen values, unseen wording and copying, and keep sampled mixed/public loss within 5%
of V22. Candidate references never roll training back or publish anything automatically.

If paused, **Run all again with this identical notebook and settings** to restore the last
confirmed checkpoint and optimizer. If finished, it displays the saved comparison.
Do not extend the run unchanged before inspecting results. One GPU is used; T4 uses FP16 AMP.

## Storage and preservation

New datasets/checkpoints/reports stay in RAM and upload directly to the PUBLIC repository
https://huggingface.co/iiidevv/nebula-raw-training under **experiments/nebula-v25-answers/**.
Existing raw/public data are referenced. Only code, notebooks and packages use disk.
No local checkpoint/data fallback, Google Drive or ZIP. Previous versions remain intact.
Wait for **v25CheckpointConfirmed** before closing. An abrupt shutdown may lose progress
since the last backup. Backup errors stop training. No website deployment is performed.
Changed settings/code require a new OUTPUT_RUN.


In [ ]:
import os, sys, subprocess
from pathlib import Path
os.environ['HF_HUB_DISABLE_XET']='1'
subprocess.run([sys.executable,'-m','pip','install','-q',
    'huggingface_hub>=0.34,<2','requests>=2.32,<3','numpy>=1.26,<3',
    'pyarrow>=18,<24','fsspec>=2024.6,<2027'],check=True)
from huggingface_hub import constants
constants.HF_HUB_DISABLE_XET=True
# Keep the platform's existing PyTorch/CUDA installation (Runpod PyTorch 2.8 is suitable).
token=os.environ.get('NEBULA_HF_TOKEN') or os.environ.get('HF_TOKEN')
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token=UserSecretsClient().get_secret('NEBULA_HF_TOKEN')
    except Exception:
        raise RuntimeError('Add and enable Kaggle secret NEBULA_HF_TOKEN, then Run all again.') from None
if not token or not token.strip():
    raise RuntimeError('Kaggle secret NEBULA_HF_TOKEN is empty.')
os.environ['NEBULA_HF_TOKEN']=token.strip()
del token


In [ ]:
OUTPUT_RUN='nebula-v25-answers'
CONFIG={'sourceRepo': 'iiidevv/nebula-raw-training', 'sourceRun': 'nebula-v22-conversation', 'sourceRevision': '6c670c3670444d7cde2b17760fea1cdde4a93b18', 'expectedSourceUpdates': 4000, 'rawCorpusRun': 'nebula-v18-raw-1gb', 'rawCorpusRevision': 'e167fff3bb27c36c25460fcd4c61915a5b7016ed', 'updates': 1000, 'seed': 230221, 'learningRate': 3e-06, 'warmup': 60, 'batch': 8, 'microbatch': 1, 'saveEvery': 100, 'evaluateEvery': 500, 'publicSourceRun': 'nebula-v23-longrun', 'publicSourceRevision': 'c21aafa220d33b9787791fcaf7aebf413bc62cf8', 'maxSessionHours': 7.5}


In [ ]:
sources={'nebula.py': '"""Nebula\'s small, randomly initialized causal Transformer. No pretrained weights."""\nimport json\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\nimport torch\nfrom torch import nn\n\nBOS, EOS, SYSTEM, USER, ASSISTANT, PAD = range(256, 262)\nVOCAB = 262\nROLES = {"system": SYSTEM, "user": USER, "assistant": ASSISTANT}\n\n@dataclass\nclass Config:\n    width: int = 128\n    heads: int = 4\n    layers: int = 2\n    context: int = 512\n\ndef config_for_preset(preset):\n    presets = {\n        \'smoke\': Config(width=64,heads=2,layers=1,context=256),\n        \'cpu\': Config(),\n        \'cpu-medium\': Config(width=192,heads=6,layers=4,context=1024),\n        \'cpu-large\': Config(width=256,heads=8,layers=6,context=1024),\n        \'gpu\': Config(width=384,heads=8,layers=6,context=2048),\n    }\n    if preset not in presets: raise ValueError(\'Unknown model preset\')\n    return presets[preset]\n\ndef encode(messages, answer_prefix=False):\n    tokens, learn = [BOS], [False]\n    for message in messages:\n        role = message["role"]\n        tokens.append(ROLES[role]); learn.append(False)\n        content = list(message["content"].encode("utf-8"))\n        tokens.extend(content); learn.extend([role == "assistant"] * len(content))\n        tokens.append(EOS); learn.append(role == "assistant")\n    if answer_prefix:\n        tokens.append(ASSISTANT); learn.append(False)\n    return tokens, learn\n\ndef pack_prompt(messages, context):\n    systems = [m for m in messages if m[\'role\'] == \'system\']\n    history = [m for m in messages if m[\'role\'] != \'system\']\n    if not history or history[-1][\'role\'] != \'user\': raise ValueError(\'A user question is required\')\n    prefix, _ = encode(systems)\n    selected = []\n    # Keep every system instruction and the full latest question; never silently cut rules.\n    available = context - len(prefix) - 2\n    for message in reversed(history):\n        chunk = encode([message])[0][1:]\n        if len(chunk) > available:\n            if not selected: raise ValueError(\'Rules and question exceed the model context\')\n            break\n        selected.insert(0, chunk); available -= len(chunk)\n    return prefix + [token for chunk in selected for token in chunk] + [ASSISTANT]\n\nclass Nebula(nn.Module):\n    def __init__(self, config):\n        super().__init__(); self.config = config\n        self.token = nn.Embedding(VOCAB, config.width)\n        self.position = nn.Embedding(config.context, config.width)\n        self.blocks = nn.ModuleList(nn.TransformerEncoderLayer(config.width, config.heads, config.width * 4, dropout=0.0, activation="gelu", batch_first=True, norm_first=True) for _ in range(config.layers))\n        self.norm = nn.LayerNorm(config.width)\n        self.output = nn.Linear(config.width, VOCAB, bias=False)\n        self.output.weight = self.token.weight\n        self.register_buffer(\'_causal_mask\',torch.ones(config.context,config.context,dtype=torch.bool).triu(1),persistent=False)\n        self.apply(self._initialize)\n\n    @staticmethod\n    def _initialize(module):\n        if isinstance(module, (nn.Linear, nn.Embedding)):\n            nn.init.normal_(module.weight, std=0.02)\n            if isinstance(module, nn.Linear) and module.bias is not None: nn.init.zeros_(module.bias)\n\n    def forward(self, tokens):\n        length = tokens.shape[1]\n        if length > self.config.context: raise ValueError("Context is too long")\n        value = self.token(tokens) + self.position(torch.arange(length, device=tokens.device))\n        causal = self._causal_mask[:length,:length]\n        for block in self.blocks:\n            if self.training and getattr(self,\'_activation_checkpointing\',False) and torch.is_grad_enabled():\n                from torch.utils.checkpoint import checkpoint\n                value=checkpoint(block,value,src_mask=causal,use_reentrant=False)\n            else:value=block(value,src_mask=causal)\n        return self.output(self.norm(value))\n\n    @torch.inference_mode()\n    def reply(self, messages, max_tokens=160, temperature=0.7):\n        self.eval()\n        encoded = pack_prompt(messages, self.config.context)\n        device = next(self.parameters()).device\n        tokens = torch.tensor([encoded], device=device)\n        result = []\n        for _ in range(min(max_tokens, self.config.context - len(encoded))):\n            logits = self(tokens)[:, -1, :].clone()\n            end = logits[:, EOS].clone(); logits[:, BOS:] = float(\'-inf\'); logits[:, EOS] = end\n            if temperature == 0: next_token = logits.argmax(-1, keepdim=True)\n            else: next_token = torch.multinomial(torch.softmax(logits / temperature, -1), 1)\n            value = int(next_token.item())\n            if value == EOS: break\n            result.append(value); tokens = torch.cat([tokens, next_token], dim=1)\n        return bytes(result).decode(\'utf-8\', errors=\'replace\').strip()\n\ndef save(model, path, metadata, training_state=None):\n    if any(not torch.isfinite(parameter).all() for parameter in model.parameters()):\n        raise ValueError(\'Checkpoint contains non-finite weights; existing saved weights were not replaced\')\n    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(\'.tmp\')\n    payload={\'format\': 1, \'config\': asdict(model.config), \'state\': model.state_dict(), \'metadata\': metadata}\n    if training_state is not None: payload[\'trainingState\']=training_state\n    torch.save(payload, temporary)\n    temporary.replace(path)\n\ndef load(path):\n    data = torch.load(path, map_location=\'cpu\', weights_only=True)\n    if data.get(\'format\') != 1: raise ValueError(\'Unsupported model checkpoint\')\n    model = Nebula(Config(**data[\'config\'])); model.load_state_dict(data[\'state\']); model.eval()\n    if any(not torch.isfinite(parameter).all() for parameter in model.parameters()):\n        raise ValueError(\'Checkpoint contains non-finite weights; resume from a healthy best checkpoint\')\n    return model, data.get(\'metadata\', {})\n', 'v18_stream.py': '"""RAM-only raw corpus preparation and CUDA training; durable state lives on HF."""\nimport argparse\nfrom collections import OrderedDict\nimport copy\nimport fnmatch\nimport gc\nimport gzip\nimport hashlib\nimport io\nimport json\nimport math\nimport os\nimport random\nimport re\nimport time\nimport unicodedata\nfrom urllib.parse import quote\nimport numpy as np\nimport requests\nimport torch\nimport torch.nn.functional as F\nfrom nebula import Nebula,Config,BOS,EOS\nfrom v8_store import safe_server_detail\n\nFORMAT=\'nebula-v18-ram-stream-1\'\nSOURCES={\n \'fineweb\':{\'dataset\':\'HuggingFaceFW/fineweb-edu\',\'subset\':\'sample-10BT\',\'revision\':\'87f09149ef4734204d70ed1d046ddc9ca3f2b8f9\',\'license\':\'odc-by\'},\n \'cosmopedia\':{\'dataset\':\'HuggingFaceTB/smollm-corpus\',\'subset\':\'cosmopedia-v2\',\'revision\':\'3ba9d605774198c5868892d7a8deda78031a781f\',\'license\':\'odc-by\'},\n \'stories\':{\'dataset\':\'roneneldan/TinyStories\',\'subset\':\'default\',\'revision\':\'f54c09fd23315a6f9c86f9dc80f725de7d8f9c64\',\'license\':\'cdla-sharing-1.0\'},\n \'textbooks\':{\'dataset\':\'HuggingFaceTB/cosmopedia\',\'subset\':\'openstax\',\'revision\':\'0ae6ec63f91742bd2d1eaef4f02232c55d719385\',\'license\':\'apache-2.0\'}}\nDEFAULT={\'sourceRepo\':\'iiidevv/nebula-research\',\'sourceRun\':\'nebula-v17-lab\',\'sourceKey\':\'current-foundationWeights\',\n \'textBytes\':1_000_000_000,\'weights\':{\'fineweb\':50,\'cosmopedia\':25,\'stories\':15,\'textbooks\':10},\n \'seed\':1802026,\'validationPercent\':2,\'minDocumentBytes\':200,\'maxDocumentBytes\':1_000_000,\n \'shardTextBytes\':32_000_000,\'steps\':100_000,\'batch\':8,\'microbatch\':1,\'learningRate\':0.00003,\n \'warmup\':200,\'cloudSaveEvery\':500,\'evaluateEvery\':1000,\'shardHoldSteps\':64}\nSECRET=re.compile(r\'\\b(?:hf_|gsk_|sk-|AIza|AQ\\.)[\\w-]{18,}|\\bBearer\\s+[\\w.~/+=-]{18,}\')\nPROBES=[\'A child found a small boat beside the pond.\',\'Ice melts when it takes in heat.\',\n        \'The rain stopped, and the neighbors opened their windows.\']\n\n\ndef emit(**value):print(json.dumps(value),flush=True)\ndef sha(value):return hashlib.sha256(value).hexdigest()\ndef digest(value):return sha(json.dumps(value,sort_keys=True,separators=(\',\',\':\')).encode())\ndef engine_hash():\n    # Source code is allowed on disk; corpus/model/optimizer artifacts never use local files.\n    from pathlib import Path\n    return sha((Path(__file__).read_text(encoding=\'utf-8\')+Path(__file__).with_name(\'nebula.py\').read_text(encoding=\'utf-8\')).encode())\n\n\ndef validate(config):\n    if set(config)!=set(DEFAULT):raise ValueError(\'Use exactly the documented CONFIG keys\')\n    if not re.fullmatch(r\'[\\w-]+/[\\w.-]+\',config[\'sourceRepo\']) or not re.fullmatch(r\'nebula-[\\w-]{1,55}\',config[\'sourceRun\']) or not re.fullmatch(r\'[a-zA-Z][\\w-]{0,90}\',config[\'sourceKey\']):\n        raise ValueError(\'Invalid source reference\')\n    bounds={\'textBytes\':(10000,4_000_000_000),\'seed\':(0,2**31),\'validationPercent\':(1,10),\n       \'minDocumentBytes\':(32,10000),\'maxDocumentBytes\':(1000,1_000_000),\'shardTextBytes\':(1000,64_000_000),\n       \'steps\':(1,1_000_000),\'batch\':(1,32),\'microbatch\':(1,8),\'warmup\':(1,5000),\n       \'cloudSaveEvery\':(1,5000),\'evaluateEvery\':(1,10000),\'shardHoldSteps\':(1,256)}\n    for key,(lo,hi) in bounds.items():\n        if type(config[key]) is not int or not lo<=config[key]<=hi:raise ValueError(\'Invalid \'+key)\n    mix=config[\'weights\']\n    if set(mix)!=set(SOURCES) or any(type(v) is not int or not 0<=v<=100 for v in mix.values()) or sum(mix.values())!=100:raise ValueError(\'Data percentages must total 100\')\n    if config[\'microbatch\']>config[\'batch\'] or config[\'minDocumentBytes\']>config[\'maxDocumentBytes\']:raise ValueError(\'Invalid batch/document limits\')\n    if type(config[\'learningRate\']) not in (int,float) or not math.isfinite(config[\'learningRate\']) or not 0<config[\'learningRate\']<=.001:raise ValueError(\'Invalid learning rate\')\n    return config\n\n\nclass Hub:\n    """Explicit byte transfers: no hf_hub_download, filesystem cache or temporary data files."""\n    def __init__(self,repo,token,run):\n        from huggingface_hub import HfApi\n        if not re.fullmatch(r\'[\\w-]+/[\\w.-]+\',repo) or not re.fullmatch(r\'nebula-v18-[\\w-]{1,45}\',run):raise ValueError(\'Invalid public destination/run\')\n        self.repo=repo;self.token=token;self.prefix=\'experiments/\'+run+\'/\'\n        self.api=HfApi(token=token);self.session=requests.Session()\n        if self.api.whoami()[\'name\']!=repo.split(\'/\')[0]:raise ValueError(\'Use your own public output namespace\')\n        self.api.create_repo(repo,repo_type=\'model\',private=False,exist_ok=True)\n        info=self.api.model_info(repo)\n        if info.private is not False:raise ValueError(\'V18 output must be a separate public repository\')\n        self.revision=info.sha\n\n    def fetch(self,repo,path,revision,*,dataset=False):\n        prefix=\'datasets/\' if dataset else \'\'\n        if not re.fullmatch(r\'[\\w-]+/[\\w.-]+\',repo) or \'..\' in path.split(\'/\') or path.startswith(\'/\') or \'\\\\\' in path:\n            raise ValueError(\'Invalid remote file reference\')\n        url=\'https://huggingface.co/\'+prefix+repo+\'/resolve/\'+quote(revision,safe=\'\')+\'/\'+quote(path,safe=\'/\')\n        for attempt in range(4):\n            response=self.session.get(url,headers={\'Authorization\':\'Bearer \'+self.token},timeout=(30,180))\n            if response.status_code in [408,429,500,502,503,504] and attempt<3:\n                response.close();time.sleep(min(60,5*2**attempt));continue\n            response.raise_for_status();return response.content\n        raise RuntimeError(\'Remote read failed\')\n\n    def state(self):\n        try:return json.loads(self.fetch(self.repo,self.prefix+\'state.json\',self.revision))\n        except requests.HTTPError as error:\n            if error.response.status_code==404:return None\n            raise\n\n    def read_entry(self,entry,repo=None,revision=None,prefix=None):\n        repo=repo or self.repo;revision=revision or self.revision;prefix=prefix or self.prefix\n        if type(entry.get(\'bytes\')) is not int or not 0<entry[\'bytes\']<=2_147_483_648:raise ValueError(\'Invalid object size\')\n        parts=entry.get(\'parts\',[entry])\n        if not 1<=len(parts)<=32 or sum(p[\'bytes\'] for p in parts)!=entry[\'bytes\']:raise ValueError(\'Invalid object parts\')\n        data=bytearray()\n        for part in parts:\n            if not part[\'path\'].startswith(prefix):raise ValueError(\'Unexpected object namespace\')\n            value=self.fetch(repo,part[\'path\'],revision)\n            if len(value)!=part[\'bytes\'] or sha(value)!=part[\'sha256\']:raise ValueError(\'Object part checksum mismatch\')\n            data.extend(value)\n        if sha(data)!=entry[\'sha256\']:raise ValueError(\'Object checksum mismatch\')\n        return data\n\n    def publish(self,state,blobs=None,root_readme=None):\n        from huggingface_hub import CommitOperationAdd\n        result=copy.deepcopy(state);operations=[];blobs=blobs or {}\n        for key,(name,blob) in blobs.items():\n            if \'..\' in name.split(\'/\') or name.startswith(\'/\') or \'\\\\\' in name:raise ValueError(\'Invalid output name\')\n            size=len(blob)\n            if not 0<size<=2_147_483_648:raise ValueError(\'Invalid output object size\')\n            identifier=sha(blob);parts=[]\n            for offset in range(0,size,64*1024*1024):\n                chunk=bytes(memoryview(blob)[offset:offset+64*1024*1024])\n                remote=self.prefix+\'objects/\'+identifier+\'/\'+name+f\'.part{len(parts):03d}.bin\'\n                parts.append({\'path\':remote,\'bytes\':len(chunk),\'sha256\':sha(chunk)})\n                operations.append(CommitOperationAdd(path_in_repo=remote,path_or_fileobj=chunk))\n            result[key]={\'bytes\':size,\'sha256\':identifier,\'parts\':parts}\n        # Raw object pointers added by the caller are resolved before this small atomic manifest.\n        for split in [\'train\',\'validation\',\'index\']:\n            key=\'_new_\'+split\n            if key in result:\n                result[\'corpus\'][split].append(result.pop(key))\n        operations.append(CommitOperationAdd(path_in_repo=self.prefix+\'state.json\',path_or_fileobj=json.dumps(result,indent=2).encode()))\n        if root_readme:operations.append(CommitOperationAdd(path_in_repo=\'README.md\',path_or_fileobj=root_readme.encode()))\n        for attempt in range(4):\n            try:\n                saved=self.api.create_commit(repo_id=self.repo,parent_commit=self.revision,operations=operations,commit_message=\'Save Nebula RAM-only raw experiment\')\n                break\n            except Exception as error:\n                response=getattr(error,\'response\',None);status=getattr(response,\'status_code\',None)\n                retry=status in [408,429,500,502,503,504] and attempt<3\n                emit(v18BackupFailed=True,httpStatus=status,serverDetail=safe_server_detail(error),retrying=retry)\n                if not retry:raise\n                time.sleep(min(60,5*2**attempt))\n        self.revision=saved.oid\n        if json.loads(self.fetch(self.repo,self.prefix+\'state.json\',self.revision))!=result:raise ValueError(\'Remote manifest verification failed\')\n        emit(v18PublicBackupConfirmed=True,revision=self.revision,uniqueTextBytes=result.get(\'uniqueTextBytes\',0),\n             updates=result.get(\'updates\',0),corpusComplete=result.get(\'corpusComplete\',False),publicWebsiteUpdated=False)\n        return result\n\n\ndef source_paths(hub,key):\n    spec=SOURCES[key];info=hub.api.dataset_info(spec[\'dataset\'],revision=spec[\'revision\'])\n    card=info.card_data.to_dict() if hasattr(info.card_data,\'to_dict\') else dict(info.card_data or {})\n    selected=next((c for c in card.get(\'configs\',[]) if c.get(\'config_name\')==spec[\'subset\']),None)\n    patterns=[]\n    if selected:\n        for item in selected.get(\'data_files\',[]):\n            if isinstance(item,dict) and item.get(\'split\')==\'train\':\n                value=item[\'path\'];patterns.extend(value if isinstance(value,list) else [value])\n    elif spec[\'subset\']==\'default\':patterns=[\'data/train-*.parquet\']\n    if not patterns:raise ValueError(\'Pinned source has no declared train-file pattern: \'+key)\n    paths=[name for name in hub.api.list_repo_files(spec[\'dataset\'],repo_type=\'dataset\',revision=spec[\'revision\'])\n           if name.endswith(\'.parquet\') and any(fnmatch.fnmatch(name,p) for p in patterns)]\n    if not paths:raise ValueError(\'No pinned parquet training files found: \'+key)\n    return sorted(paths)\n\n\ndef source_rows(hub,key,cursor,seed):\n    from huggingface_hub import HfFileSystem\n    import pyarrow.parquet as pq\n    spec=SOURCES[key];paths=source_paths(hub,key)\n    random.Random(seed+sum(key.encode())).shuffle(paths)\n    fs=HfFileSystem(token=hub.token)\n    for file_index in range(cursor[\'file\'],len(paths)):\n        remote=f"datasets/{spec[\'dataset\']}@{spec[\'revision\']}/{paths[file_index]}"\n        start=cursor[\'row\'] if file_index==cursor[\'file\'] else 0\n        # HfFileSystem\'s fsspec block cache lives in RAM. No local parquet/cache files.\n        with fs.open(remote,\'rb\',block_size=4*1024*1024,cache_type=\'bytes\') as stream:\n            parquet=pq.ParquetFile(stream);columns=[c for c in [\'text\',\'id\',\'url\',\'file_path\',\'audience\',\'format\'] if c in parquet.schema.names]\n            if \'text\' not in columns:raise ValueError(\'Pinned source is missing raw text: \'+key)\n            position=0\n            for group in range(parquet.num_row_groups):\n                size=parquet.metadata.row_group(group).num_rows\n                if position+size<=start:position+=size;continue\n                for batch in parquet.iter_batches(batch_size=32,row_groups=[group],columns=columns):\n                    for row in batch.to_pylist():\n                        index=position;position+=1\n                        if index<start:continue\n                        yield row,{\'file\':file_index,\'row\':index+1},paths[file_index]\n    # Exhaustion is checked against the requested source budget by prepare().\n\n\ndef convert(row,config):\n    text=row.get(\'text\')\n    if not isinstance(text,str):return None\n    text=text.replace(\'\\r\\n\',\'\\n\').replace(\'\\r\',\'\\n\').strip()\n    try:content=text.encode(\'utf-8\')\n    except UnicodeError:return None\n    if not config[\'minDocumentBytes\']<=len(content)<=config[\'maxDocumentBytes\']:return None\n    if \'\\ufffd\' in text or any(ord(c)<32 and c not in \'\\n\\t\' for c in text) or SECRET.search(text):return None\n    if any(p.casefold() in text.casefold() for p in PROBES):return None\n    identifier=sha(\' \'.join(unicodedata.normalize(\'NFKC\',text).casefold().split()).encode())\n    return content,identifier\n\n\ndef prepare(hub,state,hours,rows_fn=source_rows):\n    config=state[\'config\'];state=copy.deepcopy(state);seen=set()\n    for entry in state[\'corpus\'][\'index\']:\n        for line in gzip.decompress(hub.read_entry(entry)).splitlines():seen.add(json.loads(line)[\'hash\'])\n    buffers={\'train\':bytearray(),\'validation\':bytearray()};records=[];pending_bytes=0\n    started=time.monotonic();last_log=state[\'uniqueTextBytes\']\n    def flush():\n        nonlocal state,pending_bytes,records,buffers\n        blobs={}\n        for split in buffers:\n            if buffers[split]:blobs[\'_new_\'+split]=(split+\'.bin\',buffers[split])\n        if records:blobs[\'_new_index\']=(\'public-index.jsonl.gz\',gzip.compress(b\'\\n\'.join(json.dumps(r).encode() for r in records),mtime=0))\n        state=hub.publish(state,blobs)\n        buffers={\'train\':bytearray(),\'validation\':bytearray()};records=[];pending_bytes=0\n    try:\n        for key in SOURCES:\n            goal=config[\'textBytes\']*config[\'weights\'][key]//100\n            if state[\'sourceBytes\'][key]>=goal*.995:continue\n            emit(v18Acquiring=key,goalUniqueBytes=goal,resumeCursor=state[\'cursors\'][key])\n            for row,cursor,path in rows_fn(hub,key,state[\'cursors\'][key],config[\'seed\']):\n                if time.monotonic()-started>=hours*3600:flush();emit(v18PreparationPaused=True);return state\n                converted=convert(row,config)\n                state[\'cursors\'][key]=cursor\n                if converted:\n                    content,identifier=converted\n                    if identifier not in seen and len(content)<=goal-state[\'sourceBytes\'][key]:\n                        split=\'validation\' if int(identifier[:8],16)%100<config[\'validationPercent\'] else \'train\'\n                        tokens=np.empty(len(content)+2,dtype=\'<u2\');tokens[0]=BOS;tokens[-1]=EOS;tokens[1:-1]=np.frombuffer(content,dtype=np.uint8)\n                        offset=len(buffers[split])//2;buffers[split].extend(tokens.tobytes())\n                        provenance={k:str(row[k])[:1000] for k in [\'id\',\'url\',\'file_path\',\'audience\',\'format\'] if k in row}\n                        if SECRET.search(json.dumps(provenance)):raise ValueError(\'Credential pattern in upstream attribution\')\n                        records.append({\'hash\':identifier,\'source\':key,\'sourceFile\':path,\'sourceRow\':cursor[\'row\']-1,\n                            \'split\':split,\'shard\':len(state[\'corpus\'][split]),\'startToken\':offset,\'endToken\':len(buffers[split])//2,\n                            \'utf8Bytes\':len(content),\'provenance\':provenance})\n                        seen.add(identifier);state[\'sourceBytes\'][key]+=len(content);state[\'uniqueTextBytes\']+=len(content)\n                        state[\'documents\']+=1;pending_bytes+=len(content)\n                if state[\'uniqueTextBytes\']-last_log>=1_000_000:\n                    last_log=state[\'uniqueTextBytes\'];emit(v18PreparedBytes=last_log,source=key,documents=state[\'documents\'])\n                if pending_bytes>=config[\'shardTextBytes\']:flush()\n                if state[\'sourceBytes\'][key]>=goal*.995 or goal-state[\'sourceBytes\'][key]<config[\'minDocumentBytes\']:break\n            if state[\'sourceBytes\'][key]<goal*.90:flush();raise ValueError(\'Source exhausted below budget: \'+key)\n        state[\'corpusComplete\']=True;flush();return state\n    except KeyboardInterrupt:\n        flush();emit(v18PreparationPaused=True);return state\n\n\nclass RemotePool:\n    def __init__(self,hub,entries,context,hold=64):\n        self.hub=hub;self.entries=[e for e in entries if e[\'bytes\']//2>context];self.context=context;self.hold=hold\n        self.cache=OrderedDict()\n        if not self.entries:raise ValueError(\'Not enough packed tokens for a full context window\')\n        self.weights=[e[\'bytes\']//2-context for e in self.entries]\n    def data(self,index):\n        if index not in self.cache:\n            self.cache[index]=np.frombuffer(self.hub.read_entry(self.entries[index]),dtype=\'<u2\')\n            if len(self.cache)>2:self.cache.popitem(last=False)\n        self.cache.move_to_end(index);return self.cache[index]\n    def batch(self,step,batch,seed):\n        # Shard locality bounds network traffic; selection is weighted by valid start positions.\n        index=random.Random(seed+step//self.hold).choices(range(len(self.entries)),weights=self.weights,k=1)[0]\n        data=self.data(index);rng=random.Random(seed+step*7919);result=[]\n        for _ in range(batch):\n            start=rng.randrange(len(data)-self.context)\n            values=torch.from_numpy(np.array(data[start:start+self.context+1],dtype=np.int64))\n            result.append((values[:-1],values[1:]))\n        return result\n\n\ndef serialize(model,metadata,training=None):\n    if any(not torch.isfinite(p).all() for p in model.parameters()):raise ValueError(\'Non-finite weights cannot replace a healthy public checkpoint\')\n    stream=io.BytesIO()\n    payload={\'format\':1,\'config\':vars(model.config),\'state\':model.state_dict(),\'metadata\':metadata}\n    if training is not None:payload[\'trainingState\']=training\n    torch.save(payload,stream)\n    return stream\n\n\ndef deserialize(blob):\n    payload=torch.load(io.BytesIO(blob),map_location=\'cpu\',weights_only=True)\n    if payload.get(\'format\')!=1:raise ValueError(\'Unsupported Nebula checkpoint\')\n    model=Nebula(Config(**payload[\'config\']));model.load_state_dict(payload[\'state\'])\n    if any(not torch.isfinite(p).all() for p in model.parameters()):raise ValueError(\'Non-finite source weights\')\n    return model,payload\n\n\n@torch.inference_mode()\ndef evaluate(model,pool):\n    emit(v18EvaluationPhase=\'heldout-loss-and-generation\')\n    device=next(model.parameters()).device;model.eval();total=0.;n=0\n    for i in range(24):\n        x,y=pool.batch(i,1,880019)[0];logits=model(x[None].to(device))\n        total+=float(F.cross_entropy(logits.flatten(0,1),y.to(device),reduction=\'sum\'));n+=len(y)\n        if (i+1)%8==0:emit(v18EvaluationLossBlocks=i+1,total=24)\n    raw=[]\n    for prefix in PROBES:\n        tokens=[BOS]+list(prefix.encode())\n        for _ in range(128):\n            value=int(model(torch.tensor([tokens[-model.config.context:]],device=device))[0,-1].argmax())\n            if value>=256:break\n            tokens.append(value)\n        raw.append({\'prefix\':prefix,\'continuation\':bytes(t for t in tokens[1+len(prefix.encode()):] if t<256).decode(\'utf-8\',errors=\'replace\'),\'grade\':\'pending-human-review\'})\n        emit(v18RawProbesComplete=len(raw),total=3)\n    chat=[]\n    for prompt in [\'Hello! What can you help me with?\',\'Explain why ice melts.\',\'Write a thank-you note to a neighbor.\',\'Which should I choose?\']:\n        answer=model.reply([{\'role\':\'system\',\'content\':\'You are Nebula AI.\'},{\'role\':\'user\',\'content\':prompt}],max_tokens=128,temperature=0)\n        chat.append({\'prompt\':prompt,\'reply\':answer,\'grade\':\'pending-human-review\'})\n        emit(v18ChatProbesComplete=len(chat),total=4)\n    return {\'validationLoss\':total/n,\'validationTargets\':n,\'rawCompletions\':raw,\'openAnswers\':chat,\n            \'qualityApproved\':False,\'humanVerified\':False,\'publicWebsiteUpdated\':False}\n\n\ndef train(hub,state,hours,action,device=\'cuda\'):\n    config=state[\'config\'];state=copy.deepcopy(state)\n    if not state[\'corpusComplete\']:raise ValueError(\'Finish ACTION=prepare before using the GPU\')\n    if device==\'cuda\' and not torch.cuda.is_available():raise RuntimeError(\'CUDA GPU not detected. No CPU training fallback was started.\')\n    if not state.get(\'source\'):\n        info=hub.api.model_info(config[\'sourceRepo\'])\n        prefix=\'experiments/\'+config[\'sourceRun\']+\'/\'\n        old=json.loads(hub.fetch(config[\'sourceRepo\'],prefix+\'state.json\',info.sha))\n        entry=old.get(config[\'sourceKey\'])\n        if not entry:raise ValueError(\'Source checkpoint unavailable; no random restart was performed\')\n        model,payload=deserialize(hub.read_entry(entry,config[\'sourceRepo\'],info.sha,prefix));del payload\n        stream=serialize(model,{\'experimental\':True,\'qualityApproved\':False,\'originWeightSha256\':entry[\'sha256\']})\n        try:state=hub.publish({**state,\'sourceSha256\':entry[\'sha256\']},{\'source\':(\'source.pt\',stream.getbuffer())})\n        finally:stream.close()\n    else:model,payload=deserialize(hub.read_entry(state.get(\'working\') or state[\'source\']))\n    model.to(device);torch.set_num_threads(2)\n    dtype=torch.bfloat16 if device==\'cuda\' and torch.cuda.is_bf16_supported() else torch.float16\n    amp=device==\'cuda\';scaler=torch.amp.GradScaler(\'cuda\',enabled=amp and dtype==torch.float16,init_scale=1024)\n    optimizer=torch.optim.AdamW(model.parameters(),lr=config[\'learningRate\'],weight_decay=.01)\n    signature=digest({\'config\':config,\'engine\':state[\'engineSha256\'],\'corpus\':state[\'corpus\'],\'source\':state[\'sourceSha256\']})\n    updates=targets=0\n    if state.get(\'working\'):\n        training=payload[\'trainingState\']\n        if training[\'signature\']!=signature:raise ValueError(\'Checkpoint/config mismatch; use a new run name for changes\')\n        optimizer.load_state_dict(training[\'optimizer\'])\n        if scaler.is_enabled() and training[\'scaler\']:scaler.load_state_dict(training[\'scaler\'])\n        elif scaler.is_enabled():emit(v18ScalerReset=True,reason=\'Source precision used no gradient scaler; optimizer state retained\')\n        updates=training[\'updates\'];targets=training[\'targets\'];del payload,training\n    elif \'payload\' in locals():del payload\n    pool=RemotePool(hub,state[\'corpus\'][\'train\'],model.config.context,config[\'shardHoldSteps\'])\n    validation=RemotePool(hub,state[\'corpus\'][\'validation\'],model.config.context,1)\n    emit(v18TrainingDevice=torch.cuda.get_device_name(0) if amp else \'CPU test\',parameters=sum(p.numel() for p in model.parameters()),\n         optimizerRestored=bool(state.get(\'working\')),uniqueCorpusBytes=state[\'uniqueTextBytes\'],precision=str(dtype) if amp else \'fp32\',\n         plannedTrainingBytePositions=config[\'steps\']*config[\'batch\']*model.config.context)\n    def report(label):\n        nonlocal state\n        result=evaluate(model,validation)\n        state=hub.publish({**state,\'latestEvaluation\':{\'label\':label,\'validationLoss\':result[\'validationLoss\']}},\n                          {\'evaluation\':(\'evaluation.json\',json.dumps(result).encode())})\n        emit(v18Evaluation=result)\n    if action==\'evaluate\':report(\'manual-\'+str(updates));return state\n    if action==\'train\' and not state.get(\'baselineEvaluated\'):\n        report(\'baseline\');state=hub.publish({**state,\'baselineEvaluated\':True})\n    started=time.monotonic();start_targets=targets;start_updates=updates;last_backup=updates;bad=0\n    budget=min(config[\'steps\'],updates+100) if action==\'benchmark\' else config[\'steps\']\n    def backup():\n        nonlocal state,last_backup\n        if updates==last_backup and state.get(\'working\'):return\n        stream=serialize(model,{\'updates\':updates,\'supervisedBytePositions\':targets,\'qualityApproved\':False,\'experimental\':True},\n              {\'signature\':signature,\'optimizer\':optimizer.state_dict(),\'scaler\':scaler.state_dict(),\'updates\':updates,\'targets\':targets})\n        try:state=hub.publish({**state,\'updates\':updates,\'trainingBytePositions\':targets}, {\'working\':(\'working.pt\',stream.getbuffer())})\n        finally:stream.close()\n        last_backup=updates\n    interrupted=False\n    try:\n        while updates<budget and time.monotonic()-started<hours*3600:\n            pairs=pool.batch(updates,config[\'batch\'],config[\'seed\']);model.train();optimizer.zero_grad(set_to_none=True)\n            rate=config[\'learningRate\']*min(1.,(updates+1)/config[\'warmup\'])*(.1+.9*.5*(1+math.cos(math.pi*updates/config[\'steps\'])))\n            for group in optimizer.param_groups:group[\'lr\']=rate\n            loss_value=0.;denominator=sum(len(y) for _,y in pairs)\n            for offset in range(0,len(pairs),config[\'microbatch\']):\n                micro=pairs[offset:offset+config[\'microbatch\']]\n                x=torch.stack([p[0] for p in micro]).to(device);y=torch.stack([p[1] for p in micro]).to(device)\n                with torch.autocast(\'cuda\',dtype=dtype,enabled=amp):\n                    objective=F.cross_entropy(model(x).flatten(0,1),y.flatten(),reduction=\'sum\')/denominator\n                if not torch.isfinite(objective):raise ValueError(\'Non-finite loss; previous public checkpoint retained\')\n                loss_value+=float(objective.detach());scaler.scale(objective).backward()\n            scaler.unscale_(optimizer);norm=torch.nn.utils.clip_grad_norm_(model.parameters(),1.)\n            if not torch.isfinite(norm):\n                bad+=1\n                if not scaler.is_enabled() or bad>12:raise ValueError(\'Non-finite gradients; previous checkpoint retained\')\n                scaler.update(new_scale=scaler.get_scale()/2);continue\n            bad=0;scaler.step(optimizer);scaler.update();updates+=1;targets+=denominator\n            if updates%25==0:emit(v18Update=updates,budget=config[\'steps\'],loss=round(loss_value,4),learningRate=rate,\n                supervisedBytePositions=targets,bytePositionsPerSecond=round((targets-start_targets)/max(1,time.monotonic()-started)))\n            if action==\'train\' and updates%config[\'cloudSaveEvery\']==0:\n                backup()\n                if updates%config[\'evaluateEvery\']==0:report(\'update-\'+str(updates))\n    except KeyboardInterrupt:interrupted=True\n    if action==\'benchmark\':\n        elapsed=time.monotonic()-started;rate=(targets-start_targets)/max(.01,elapsed)\n        result={\'benchmarkUpdates\':updates-start_updates,\'seconds\':elapsed,\'bytePositionsPerSecond\':rate,\n                \'estimatedComputeHoursForRemainingBudget\':(config[\'steps\']-start_updates)*config[\'batch\']*model.config.context/max(1,rate)/3600,\n                \'weightsDiscarded\':True,\'note\':\'Includes corpus downloads; excludes remote full-checkpoint saves and evaluation.\'}\n        state=hub.publish(state,{\'benchmark\':(\'benchmark.json\',json.dumps(result).encode())});emit(v18Benchmark=result)\n    else:\n        backup();report(\'session-\'+str(updates))\n        state=hub.publish({**state,\'trainingComplete\':updates>=config[\'steps\']})\n    emit(v18SessionFinished=True,interrupted=interrupted,updatesSaved=state.get(\'updates\',0),publicWebsiteUpdated=False)\n    del optimizer,model,pool,validation;gc.collect()\n    if torch.cuda.is_available():torch.cuda.empty_cache()\n    return state\n\n\ndef readme(repo):\n    attribution=\'\\n\'.join(f"- {key}: [{s[\'dataset\']}](https://huggingface.co/datasets/{s[\'dataset\']}/tree/{s[\'revision\']}), `{s[\'subset\']}`, `{s[\'license\']}`." for key,s in SOURCES.items())\n    return f\'\'\'---\nlicense: other\nlicense_name: per-source-data-licenses\nlicense_link: https://huggingface.co/{repo}/blob/main/README.md#licenses\nlibrary_name: pytorch\ntags:\n- nebula\n- experimental\n---\n# Nebula V18 — raw text and GPU experiment\n\nPublic experimental continuation of the owner\'s Nebula weights. Durable datasets, packed byte-token shards,\nsource indexes, model/optimizer/scaler checkpoints and reports are uploaded directly from RAM.\nNo dataset or checkpoint is written to the worker\'s filesystem. Python code, installed packages and\nthe notebook itself can exist on disk. This runner does not ingest account chats or review feeds.\nInherited weights and public web data are not guaranteed free of personal information.\nNo website deployment or automatic quality approval is enabled.\n\n## Licenses\n\nEach corpus retains its upstream terms and attribution; this collection does not relicense it.\nPinned upstream cards are stored with each run; original FineWeb document URLs are retained in compressed indexes.\nThe new textbooks component is synthetic Cosmopedia text seeded from OpenStax outlines.\nODC-By: https://opendatacommons.org/licenses/by/1-0/\nCDLA-Sharing-1.0: https://cdla.dev/sharing-1-0/\nApache-2.0: https://www.apache.org/licenses/LICENSE-2.0\n\n{attribution}\n\n## Representation and limitations\n\nPacked tokens are little-endian uint16: byte IDs 0–255, BOS=256, EOS=257. Whole UTF-8 documents\nare normalized-exact-deduplicated and assigned by hash to train or validation before packing.\nCredential-pattern/control/size filters are applied; this is not a full PII, factual or fuzzy-duplicate audit.\nThe model may have seen overlapping public text before this experiment.\nSource mix percentages describe acquired UTF-8 bytes; training samples packed positions, including repeated exposure.\nTraining batches use deterministic shard locality (64 updates per sampled shard) to bound network traffic.\nAt most two training shards and two validation shards are cached in RAM; deduplication hashes also use RAM.\nFull optimizer snapshots consume substantial host RAM and public storage; GPU memory alone is not sufficient.\nOnly confirmed public checkpoints survive worker loss. The history is retained and public storage is not unlimited.\n\'\'\'\n\n\ndef run(repo=\'iiidevv/nebula-raw-training\',run_name=\'nebula-v18-raw-1gb\',config=None,action=\'prepare\',hours=2,\n        *,hub_factory=Hub,rows_fn=source_rows,device=\'cuda\'):\n    config=validate(copy.deepcopy(DEFAULT if config is None else config))\n    if action not in [\'prepare\',\'benchmark\',\'train\',\'evaluate\'] or not 0<hours<=12:raise ValueError(\'Invalid action/session hours\')\n    if action!=\'prepare\' and device==\'cuda\' and not torch.cuda.is_available():raise RuntimeError(\'CUDA GPU not detected; no CPU training fallback started\')\n    if (repo,run_name)==(config[\'sourceRepo\'],config[\'sourceRun\']):raise ValueError(\'Do not overwrite the source experiment\')\n    token=os.environ.get(\'NEBULA_HF_TOKEN\') or os.environ.get(\'HF_TOKEN\')\n    if not token:raise RuntimeError(\'Set NEBULA_HF_TOKEN in the platform secret/environment settings\')\n    hub=hub_factory(repo,token,run_name);state=hub.state();code=engine_hash()\n    if state is None:\n        state={\'format\':FORMAT,\'config\':config,\'engineSha256\':code,\'sources\':SOURCES,\'uniqueTextBytes\':0,\'documents\':0,\n          \'sourceBytes\':{k:0 for k in SOURCES},\'cursors\':{k:{\'file\':0,\'row\':0} for k in SOURCES},\n          \'corpus\':{\'train\':[],\'validation\':[],\'index\':[]},\'corpusComplete\':False,\'updates\':0,\'qualityApproved\':False,\'publicAutoUpdate\':False}\n        blobs={}\n        for key,spec in SOURCES.items():blobs[\'card-\'+key]=(key+\'-README.md\',hub.fetch(spec[\'dataset\'],\'README.md\',spec[\'revision\'],dataset=True))\n        state=hub.publish(state,blobs,root_readme=readme(repo))\n    if state[\'format\']!=FORMAT or state[\'config\']!=config or state[\'sources\']!=SOURCES or state[\'engineSha256\']!=code:\n        raise ValueError(\'Run settings/code changed. Keep the original settings to resume or use a NEW RUN_NAME\')\n    if action==\'prepare\':\n        if not state[\'corpusComplete\']:state=prepare(hub,state,hours,rows_fn)\n        emit(v18DataReady=state[\'corpusComplete\'],uniqueUtf8Bytes=state[\'uniqueTextBytes\'],trainingStarted=False)\n        return state\n    return train(hub,state,hours,action,device)\n\n\ndef main():\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--action\',choices=[\'prepare\',\'benchmark\',\'train\',\'evaluate\'],default=\'prepare\')\n    parser.add_argument(\'--repo\',default=\'iiidevv/nebula-raw-training\');parser.add_argument(\'--run-name\',default=\'nebula-v18-raw-1gb\')\n    parser.add_argument(\'--hours\',type=float,default=2)\n    args=parser.parse_args();run(args.repo,args.run_name,action=args.action,hours=args.hours)\n\n\nif __name__==\'__main__\':main()\n', 'v8_store.py': '"""Atomic private manifests for large, sharded training checkpoints."""\nimport json\nfrom pathlib import Path\nimport shutil\nimport tempfile\nimport time\nimport os\nimport re\nfrom v4_store import Store as SmallStore\nfrom research_client import ResearchError,digests\n\nPART_BYTES=64*1024*1024\nMAX_BYTES=2*1024*1024*1024\n\ndef safe_server_detail(error):\n    """Expose the service\'s explanation, with credentials and URLs removed."""\n    detail=getattr(error,\'server_message\',None)\n    if not isinstance(detail,str) or not detail.strip():\n        try:\n            body=error.response.json()\n            detail=body.get(\'error\') or body.get(\'message\') if isinstance(body,dict) else None\n        except Exception:detail=None\n    if not isinstance(detail,str):return \'No structured server explanation was provided.\'\n    for key,value in os.environ.items():\n        if re.search(r\'TOKEN|SECRET|PASSWORD|API_KEY\',key,re.I) and len(value)>=6:\n            detail=detail.replace(value,\'[redacted]\')\n    detail=re.sub(r\'https?://\\S+\',\'[URL removed]\',detail)\n    detail=re.sub(r\'(?i)\\b(?:hf_|gsk_|AIza)[A-Za-z0-9_.-]+\',\'[redacted]\',detail)\n    detail=re.sub(r\'(?i)(?:bearer\\s+|(?:token|api[_-]?key|password|secret)\\s*[:=]\\s*)[^\\s,;]+\',\'[redacted]\',detail)\n    return \' \'.join(detail.split())[:700]\n\nclass Store(SmallStore):\n    def _commit_backup(self, operations):\n        """Retry temporary failures with the SAME parent; never overwrite another writer."""\n        for attempt in range(4):\n            try:\n                return self.api.create_commit(repo_id=self.repo,operations=operations,\n                    commit_message=\'Save Nebula 100M experiment\',parent_commit=self.revision)\n            except Exception as error:\n                response=getattr(error,\'response\',None)\n                status=getattr(response,\'status_code\',None)\n                # Do not expose exception text, response bodies, URLs, headers or credentials.\n                status=status if type(status) is int else None\n                network=any(kind.__name__ in {\'Timeout\',\'TimeoutError\',\'ConnectTimeout\',\'ReadTimeout\',\n                    \'ConnectionError\',\'ConnectError\',\'ReadError\',\'RemoteProtocolError\'} for kind in type(error).__mro__)\n                retryable=status in {408,429,500,502,503,504} or network\n                reason=(\'CONCURRENT_OR_AMBIGUOUS_COMMIT\' if status in {409,412} else\n                        \'ACCESS_DENIED\' if status in {401,403} else\n                        \'REQUEST_REJECTED\' if status in {400,413,422} else\n                        \'TEMPORARY_SERVICE_FAILURE\' if retryable else \'UNCLASSIFIED_FAILURE\')\n                retry=retryable and attempt<3\n                wait=5*(2**attempt)\n                if status==429:\n                    try:wait=max(wait,min(60,max(0,float(response.headers.get(\'Retry-After\',0)))))\n                    except (AttributeError,TypeError,ValueError):pass\n                print(json.dumps({\'backupFailed\':True,\'reason\':reason,\'httpStatus\':status,\n                    \'attempt\':attempt+1,\'retrying\':retry,\'retryInSeconds\':wait if retry else 0,\n                    \'serverDetail\':safe_server_detail(error),\n                    \'localFilesPreserved\':True}),flush=True)\n                if not retry:\n                    raise ResearchError(\'V8_BACKUP_FAILED_OR_CONCURRENT_RUN:\'+reason) from None\n                time.sleep(wait)\n\n    def download(self,file,destination):\n        if \'parts\' not in file:return super().download(file,destination)\n        if not 0<file.get(\'bytes\',0)<=MAX_BYTES or not isinstance(file.get(\'parts\'),list) or not 1<=len(file[\'parts\'])<=32 or sum(p.get(\'bytes\',0) for p in file[\'parts\'])!=file[\'bytes\']:\n            raise ResearchError(\'V8_INVALID_SHARDED_CHECKPOINT\')\n        destination=Path(destination);destination.parent.mkdir(parents=True,exist_ok=True)\n        if destination.is_file() and destination.stat().st_size==file[\'bytes\'] and digests(destination)[0]==file[\'sha256\']:return destination\n        temporary=destination.with_suffix(destination.suffix+\'.download-tmp\')\n        try:\n            with tempfile.TemporaryDirectory(dir=destination.parent,prefix=\'v8-parts-\') as folder:\n                with temporary.open(\'wb\') as output:\n                    for i,part in enumerate(file[\'parts\']):\n                        local=super().download(part,Path(folder)/str(i))\n                        with local.open(\'rb\') as source:shutil.copyfileobj(source,output,1024*1024)\n            if temporary.stat().st_size!=file[\'bytes\'] or digests(temporary)[0]!=file[\'sha256\']:raise ResearchError(\'V8_CHECKSUM_MISMATCH\')\n            temporary.replace(destination)\n        except BaseException:\n            temporary.unlink(missing_ok=True);raise\n        return destination\n\n    def publish(self,state,files,project,manifest_name=\'state.json\'):\n        from huggingface_hub import CommitOperationAdd\n        if manifest_name!=\'state.json\':raise ResearchError(\'V8_INVALID_MANIFEST\')\n        self.info();value=json.loads(json.dumps(state));operations=[]\n        with tempfile.TemporaryDirectory(dir=project,prefix=\'v8-upload-\') as folder:\n            for key,(path,name) in files.items():\n                path=Path(path);size=path.stat().st_size\n                if \'..\' in name.split(\'/\') or name.startswith(\'/\') or \'\\\\\' in name or not 0<size<=MAX_BYTES:raise ResearchError(\'V8_INVALID_FILE_REFERENCE\')\n                # Use a recognized binary extension for SQLite snapshots and shards.\n                # Local file names and existing manifests remain valid.\n                if name.endswith(\'.sqlite\'):name+=\'.bin\'\n                if size<=PART_BYTES:\n                    remote=self.prefix+name\n                    value[key]={\'path\':remote,\'bytes\':size,\'sha256\':digests(path)[0]}\n                    operations.append(CommitOperationAdd(path_in_repo=remote,path_or_fileobj=str(path)));continue\n                parts=[]\n                with path.open(\'rb\') as source:\n                    index=0\n                    while chunk:=source.read(PART_BYTES):\n                        local=Path(folder)/(key+\'-\'+str(index));local.write_bytes(chunk)\n                        remote=self.prefix+name+f\'.part{index:03d}.bin\'\n                        parts.append({\'path\':remote,\'bytes\':len(chunk),\'sha256\':digests(local)[0]})\n                        operations.append(CommitOperationAdd(path_in_repo=remote,path_or_fileobj=str(local)));index+=1\n                value[key]={\'bytes\':size,\'sha256\':digests(path)[0],\'parts\':parts}\n            manifest=Path(folder)/\'state.json\';manifest.write_text(json.dumps(value,indent=2),encoding=\'utf-8\')\n            operations.append(CommitOperationAdd(path_in_repo=self.prefix+manifest_name,path_or_fileobj=str(manifest)))\n            result=self._commit_backup(operations)\n        self.revision=result.oid\n        print(json.dumps({\'v8BackupSaved\':True,\'round\':value.get(\'round\',0),\'publicWebsiteUpdated\':False}),flush=True)\n        return value\n', 'v4_store.py': '"""Private experiment snapshots, committed atomically with a resume manifest."""\nimport hashlib\nimport json\nfrom pathlib import Path\nimport re\nimport shutil\nfrom research_client import ResearchError,digests\n\nclass Store:\n    def __init__(self,repo,token,run):\n        if not re.fullmatch(r\'[\\w-]+/[\\w.-]+\',repo) or not re.fullmatch(r\'[a-zA-Z0-9_-]{1,64}\',run) or not token.startswith(\'hf_\'):\n            raise ResearchError(\'V4_STORAGE_CONFIGURATION_MISSING\')\n        from huggingface_hub import HfApi\n        self.repo=repo;self.token=token;self.prefix=\'experiments/\'+run+\'/\'\n        self.api=HfApi(endpoint=\'https://huggingface.co\',token=token)\n        self.revision=self.info().sha\n    def info(self):\n        try:value=self.api.model_info(self.repo)\n        except Exception:raise ResearchError(\'V4_STORAGE_ACCESS_FAILED\') from None\n        if value.private is not True:raise ResearchError(\'HF_REPOSITORY_MUST_BE_PRIVATE\')\n        return value\n    def state(self,manifest_name=\'state.json\'):\n        if manifest_name not in [\'state.json\',\'prepared.json\']:raise ResearchError(\'V4_INVALID_FILE_REFERENCE\')\n        from huggingface_hub import hf_hub_download\n        from huggingface_hub.errors import EntryNotFoundError\n        try:\n            path=hf_hub_download(self.repo,self.prefix+manifest_name,revision=self.revision,token=self.token)\n            return json.loads(Path(path).read_text(encoding=\'utf-8\'))\n        except EntryNotFoundError:return None\n        except Exception:raise ResearchError(\'V4_RESUME_MANIFEST_UNREADABLE\') from None\n    def download(self,file,destination):\n        from huggingface_hub import hf_hub_download\n        if not file[\'path\'].startswith(self.prefix) or \'..\' in file[\'path\'].split(\'/\') or not 0<file[\'bytes\']<=256*1024*1024:\n            raise ResearchError(\'V4_INVALID_FILE_REFERENCE\')\n        destination=Path(destination);destination.parent.mkdir(parents=True,exist_ok=True)\n        if destination.exists() and destination.stat().st_size==file[\'bytes\'] and digests(destination)[0]==file[\'sha256\']:return destination\n        try:cached=Path(hf_hub_download(self.repo,file[\'path\'],revision=self.revision,token=self.token))\n        except Exception:raise ResearchError(\'V4_SNAPSHOT_DOWNLOAD_FAILED\') from None\n        if cached.stat().st_size!=file[\'bytes\'] or digests(cached)[0]!=file[\'sha256\']:raise ResearchError(\'V4_CHECKSUM_MISMATCH\')\n        temporary=destination.with_suffix(destination.suffix+\'.download-tmp\');shutil.copyfile(cached,temporary);temporary.replace(destination)\n        return destination\n    def publish(self,state,files,project,manifest_name=\'state.json\'):\n        if manifest_name not in [\'state.json\',\'prepared.json\']:raise ResearchError(\'V4_INVALID_FILE_REFERENCE\')\n        from huggingface_hub import CommitOperationAdd\n        self.info();operations=[];state=json.loads(json.dumps(state))\n        for key,(path,name) in files.items():\n            path=Path(path)\n            if \'..\' in name.split(\'/\') or name.startswith(\'/\') or \'\\\\\' in name:raise ResearchError(\'V4_INVALID_FILE_REFERENCE\')\n            if not 0<path.stat().st_size<=256*1024*1024:raise ResearchError(\'V4_SNAPSHOT_TOO_LARGE\')\n            remote=self.prefix+name\n            state[key]={\'path\':remote,\'bytes\':path.stat().st_size,\'sha256\':digests(path)[0]}\n            operations.append(CommitOperationAdd(path_in_repo=remote,path_or_fileobj=str(path)))\n        manifest=Path(project)/(\'v4-\'+manifest_name);manifest.write_text(json.dumps(state,indent=2),encoding=\'utf-8\')\n        operations.append(CommitOperationAdd(path_in_repo=self.prefix+manifest_name,path_or_fileobj=str(manifest)))\n        try:\n            result=self.api.create_commit(repo_id=self.repo,operations=operations,\n                commit_message=\'Save bounded Nebula V4 experiment\',parent_commit=self.revision)\n        except Exception:raise ResearchError(\'V4_BACKUP_FAILED_OR_CONCURRENT_RUN\') from None\n        self.revision=result.oid\n        print(json.dumps({\'v4BackupSaved\':True,\'round\':state.get(\'round\',0),\'publicWebsiteUpdated\':False}),flush=True)\n        return state\n', 'research_client.py': '"""Private worker protocol. Tokens never go to object-storage URLs or logs."""\nimport base64\nimport hashlib\nimport os\nimport re\nimport shutil\nfrom pathlib import Path\nfrom urllib.parse import urlparse\nimport requests\n\nclass ResearchError(RuntimeError):\n    def __init__(self,code,status=0):super().__init__(str(code));self.code=code;self.status=status\n\ndef digests(path):\n    sha=hashlib.sha256();md5=hashlib.md5(usedforsecurity=False)\n    with Path(path).open(\'rb\') as stream:\n        for chunk in iter(lambda:stream.read(1024*1024),b\'\'):sha.update(chunk);md5.update(chunk)\n    return sha.hexdigest(),base64.b64encode(md5.digest()).decode()\n\ndef storage_url(url):\n    parsed=urlparse(url)\n    if parsed.scheme!=\'https\' or parsed.username or parsed.password or not (\n            parsed.hostname==\'storage.googleapis.com\' or (parsed.hostname or \'\').endswith(\'.storage.googleapis.com\')):\n        raise ResearchError(\'UNEXPECTED_STORAGE_HOST\')\n    return url\n\nclass Client:\n    def __init__(self,api=None,token=None):\n        self.api=(api or os.environ.get(\'NEBULA_RESEARCH_API\',\'\')).rstrip(\'/\')\n        self.token=token or os.environ.get(\'NEBULA_WORKER_TOKEN\',\'\')\n        parsed=urlparse(self.api)\n        if parsed.scheme!=\'https\' or parsed.username or parsed.password or parsed.query or parsed.fragment or len(self.token)<32:\n            raise ResearchError(\'WORKER_CONFIGURATION_MISSING\')\n    def call(self,action,body=None):\n        try:\n            response=requests.request(\'GET\' if body is None else \'POST\',self.api+\'/\'+action,\n                headers={\'Authorization\':\'Bearer \'+self.token},json=body,timeout=45,allow_redirects=False)\n            value=response.json()\n        except Exception:raise ResearchError(\'COORDINATOR_UNREACHABLE\') from None\n        if response.status_code!=200:raise ResearchError(value.get(\'code\',\'COORDINATOR_FAILED\'),response.status_code)\n        return value\n    def huggingface(self,ticket):\n        repo=os.environ.get(\'NEBULA_HF_REPO\',\'\')\n        token=os.environ.get(\'NEBULA_HF_TOKEN\',\'\')\n        if (not re.fullmatch(r\'[\\w-]+/[\\w.-]+\',repo) or ticket.get(\'repo\')!=repo or\n                not re.fullmatch(r\'checkpoints/[a-f0-9-]{36}\\.pt\',ticket.get(\'path\',\'\')) or\n                not token.startswith(\'hf_\')):\n            raise ResearchError(\'HF_CONFIGURATION_MISSING\')\n        try:\n            from huggingface_hub import HfApi\n            hub=HfApi(endpoint=\'https://huggingface.co\',token=token)\n            if hub.model_info(repo).private is not True:raise ResearchError(\'HF_REPOSITORY_MUST_BE_PRIVATE\')\n            return hub,repo\n        except ResearchError:raise\n        except Exception:raise ResearchError(\'HF_ACCESS_OR_DEPENDENCY_FAILED\') from None\n    def download(self,identifier,destination):\n        ticket=self.call(\'download\',{\'id\':identifier})\n        if ticket.get(\'provider\')==\'huggingface\':return self.download_huggingface(ticket,destination)\n        url=storage_url(ticket[\'url\'])\n        path=Path(destination);path.parent.mkdir(parents=True,exist_ok=True)\n        if path.exists() and path.stat().st_size==ticket[\'bytes\'] and digests(path)[0]==ticket[\'sha256\']:return path\n        temporary=path.with_suffix(path.suffix+\'.download-tmp\');sha=hashlib.sha256();count=0\n        try:\n            with requests.get(url,stream=True,timeout=90,allow_redirects=False) as response:\n                if response.status_code!=200:raise ResearchError(\'CHECKPOINT_DOWNLOAD_FAILED\')\n                with temporary.open(\'wb\') as stream:\n                    for chunk in response.iter_content(1024*1024):\n                        count+=len(chunk)\n                        if count>ticket[\'bytes\'] or count>100*1024*1024:raise ResearchError(\'CHECKPOINT_SIZE_MISMATCH\')\n                        sha.update(chunk);stream.write(chunk)\n            if count!=ticket[\'bytes\'] or sha.hexdigest()!=ticket[\'sha256\']:raise ResearchError(\'CHECKPOINT_CHECKSUM_MISMATCH\')\n            temporary.replace(path);return path\n        except ResearchError:raise\n        except Exception:raise ResearchError(\'CHECKPOINT_DOWNLOAD_FAILED\') from None\n    def download_huggingface(self,ticket,destination):\n        hub,repo=self.huggingface(ticket)\n        if (not re.fullmatch(r\'[a-f0-9]{40}\',ticket.get(\'revision\',\'\')) or\n                not isinstance(ticket.get(\'bytes\'),int) or not 0<ticket[\'bytes\']<=100*1024*1024):\n            raise ResearchError(\'HF_INVALID_DOWNLOAD\')\n        path=Path(destination);path.parent.mkdir(parents=True,exist_ok=True)\n        if path.exists() and path.stat().st_size==ticket[\'bytes\'] and digests(path)[0]==ticket[\'sha256\']:return path\n        temporary=path.with_suffix(path.suffix+\'.download-tmp\')\n        try:\n            from huggingface_hub import hf_hub_download\n            cached=Path(hf_hub_download(repo_id=repo,filename=ticket[\'path\'],revision=ticket[\'revision\'],\n                                       token=os.environ[\'NEBULA_HF_TOKEN\'],endpoint=\'https://huggingface.co\'))\n            if cached.stat().st_size!=ticket[\'bytes\'] or digests(cached)[0]!=ticket[\'sha256\']:\n                raise ResearchError(\'CHECKPOINT_CHECKSUM_MISMATCH\')\n            shutil.copyfile(cached,temporary);temporary.replace(path);return path\n        except ResearchError:raise\n        except Exception:raise ResearchError(\'HF_DOWNLOAD_FAILED\') from None\n    def upload(self,path,kind=\'checkpoint\',parent=None,lease=None):\n        path=Path(path);sha,md5=digests(path)\n        ticket=self.call(\'upload\',{\'kind\':kind,\'bytes\':path.stat().st_size,\'sha256\':sha,\'md5\':md5,\n                                   \'parent\':parent,\'lease\':lease})\n        if ticket.get(\'provider\')==\'huggingface\':\n            hub,repo=self.huggingface(ticket)\n            try:\n                hub.upload_file(path_or_fileobj=str(path),path_in_repo=ticket[\'path\'],repo_id=repo,\n                                commit_message=\'Save private Nebula research checkpoint\')\n            except Exception:\n                try:self.call(\'finish\',{\'id\':ticket[\'id\']})\n                except Exception:raise ResearchError(\'HF_UPLOAD_FAILED\') from None\n            else:self.call(\'finish\',{\'id\':ticket[\'id\']})\n            return ticket[\'id\']\n        try:\n            with path.open(\'rb\') as stream:\n                response=requests.put(storage_url(ticket[\'url\']),data=stream,headers=ticket[\'headers\'],timeout=180,allow_redirects=False)\n                # A timeout or retry may occur after an upload succeeded. The\n                # coordinator checks the immutable object before accepting it.\n                if response.status_code not in [200,201,412]:raise ResearchError(\'CHECKPOINT_UPLOAD_FAILED\')\n        except Exception:\n            try:self.call(\'finish\',{\'id\':ticket[\'id\']})\n            except Exception:raise ResearchError(\'CHECKPOINT_UPLOAD_FAILED\') from None\n        else:self.call(\'finish\',{\'id\':ticket[\'id\']})\n        return ticket[\'id\']\n', 'v18_decode_probe.py': '"""Inference-only comparison; never updates V18 weights, optimizer or manifest."""\nimport gc\nimport json\nimport os\nimport re\nimport uuid\nfrom collections import Counter\nfrom datetime import datetime, timezone\n\nimport requests\nimport torch\nfrom nebula import BOS, EOS, pack_prompt\nfrom v18_stream import Hub, deserialize, PROBES\n\nMODES = [dict(name=\'greedy\', temperature=0., top_p=1., seeds=[0]),\n         dict(name=\'sample-0.7\', temperature=.7, top_p=.9, seeds=[17,43]),\n         dict(name=\'sample-1.0\', temperature=1., top_p=.95, seeds=[17,43])]\nQUESTIONS = [\'Hello! What can you help me with?\', \'Explain why ice melts.\',\n             \'Write a thank-you note to a neighbor.\', \'Which should I choose?\',\n             \'My name is Iris. My scarf is purple. What color is my scarf?\']\n\n\ndef repetition(text):\n    words = re.findall(r"\\w+",text.casefold())\n    grams = [tuple(words[i:i+3]) for i in range(max(0,len(words)-2))]\n    counts = Counter(grams)\n    return {\'words\':len(words), \'repeatedTrigramFraction\':\n            round(sum(n-1 for n in counts.values())/len(grams),4) if grams else 0.,\n            \'utf8ReplacementCharacters\':text.count(\'\\ufffd\')}\n\n\n@torch.inference_mode()\ndef generate(model, prompt, kind, temperature, top_p, seed, max_bytes=128):\n    if kind not in [\'raw\',\'chat\'] or not 0<=temperature<=2 or not 0<top_p<=1:\n        raise ValueError(\'Invalid generation configuration\')\n    prefix = ([BOS]+list(prompt.encode(\'utf-8\'))) if kind==\'raw\' else pack_prompt(\n        [{\'role\':\'system\',\'content\':\'You are Nebula AI.\'},{\'role\':\'user\',\'content\':prompt}],model.config.context)\n    if len(prefix)>=model.config.context:raise ValueError(\'Prompt leaves no context for generation\')\n    device=next(model.parameters()).device\n    rng=torch.Generator(device=device).manual_seed(seed)\n    tokens=torch.tensor([prefix],dtype=torch.long,device=device)\n    generated=[];stop=\'byte-budget\';model.eval()\n    limit=min(max_bytes,model.config.context-len(prefix))\n    for _ in range(limit):\n        logits=model(tokens)[0,-1].float().clone()\n        # Match the original V18 raw probe\'s stop-on-any-marker rule.\n        # Chat follows Nebula.reply: only EOS may be generated among markers.\n        if kind==\'chat\':\n            end=logits[EOS].clone();logits[BOS:]=float(\'-inf\');logits[EOS]=end\n        if temperature==0:\n            value=int(logits.argmax())\n        else:\n            values,indices=torch.sort(logits/temperature,descending=True)\n            probs=torch.softmax(values,dim=-1)\n            remove=probs.cumsum(-1)-probs>=top_p\n            values[remove]=float(\'-inf\')\n            sampled=torch.multinomial(torch.softmax(values,dim=-1),1,generator=rng)\n            value=int(indices[sampled].item())\n        if value>=256:\n            stop=\'eos\' if value==EOS else \'special-marker\';break\n        generated.append(value)\n        tokens=torch.cat([tokens,torch.tensor([[value]],device=device)],dim=1)\n    else:\n        if limit<max_bytes:stop=\'context-limit\'\n    text=bytes(generated).decode(\'utf-8\',errors=\'replace\')\n    return {\'reply\':text,\'generatedBytes\':len(generated),\'stopReason\':stop,\n            **repetition(text),\'relevanceGrade\':\'pending-human-review\'}\n\n\ndef compare(model, max_bytes=128, modes=MODES, raw=PROBES, questions=QUESTIONS):\n    answers=[]\n    total=(len(raw)+len(questions))*sum(len(m[\'seeds\']) for m in modes)\n    for kind,prompts in [(\'raw\',raw),(\'chat\',questions)]:\n        for prompt in prompts:\n            for mode in modes:\n                for seed in mode[\'seeds\']:\n                    result=generate(model,prompt,kind,mode[\'temperature\'],mode[\'top_p\'],seed,max_bytes)\n                    answer={\'kind\':kind,\'prompt\':prompt,\'mode\':mode[\'name\'],\'seed\':seed,\n                            \'temperature\':mode[\'temperature\'],\'topP\':mode[\'top_p\'],**result}\n                    answers.append(answer)\n                    print(json.dumps({\'decodeProbeProgress\':len(answers),\'total\':total,**answer}),flush=True)\n    summaries=[]\n    for kind in [\'raw\',\'chat\']:\n        for mode in modes:\n            rows=[a for a in answers if a[\'kind\']==kind and a[\'mode\']==mode[\'name\']]\n            if rows:summaries.append({\'kind\':kind,\'mode\':mode[\'name\'],\'cases\':len(rows),\n                \'meanRepeatedTrigramFraction\':sum(a[\'repeatedTrigramFraction\'] for a in rows)/len(rows),\n                \'emptyReplies\':sum(not a[\'reply\'].strip() for a in rows),\n                \'notice\':\'Less repetition does not establish correctness or relevance.\'})\n    return {\'answers\':answers,\'summary\':summaries,\'qualityApproved\':False,\'humanVerified\':False,\n            \'trainingStarted\':False,\'publicWebsiteUpdated\':False}\n\n\ndef run_probe(repo=\'iiidevv/nebula-raw-training\',run_name=\'nebula-v18-raw-1gb\',\n              revision=\'e167fff3bb27c36c25460fcd4c61915a5b7016ed\',expected_updates=10102,\n              save_report=True,max_bytes=128):\n    if not torch.cuda.is_available():raise RuntimeError(\'Enable a Kaggle NVIDIA GPU before running this probe.\')\n    if not re.fullmatch(r\'[\\w-]+/[\\w.-]+\',repo) or not re.fullmatch(r\'nebula-v18-[\\w-]{1,45}\',run_name):\n        raise ValueError(\'Invalid repository/run\')\n    token=os.environ.get(\'NEBULA_HF_TOKEN\') or os.environ.get(\'HF_TOKEN\')\n    if not token:raise RuntimeError(\'Enable Kaggle secret NEBULA_HF_TOKEN.\')\n    from huggingface_hub import HfApi,CommitOperationAdd\n    # Construct a reader without Hub.__init__: no repository creation or manifest writes.\n    hub=object.__new__(Hub);hub.repo=repo;hub.token=token;hub.prefix=\'experiments/\'+run_name+\'/\'\n    hub.api=HfApi(token=token);hub.session=requests.Session()\n    hub.revision=hub.api.model_info(repo,revision=revision or None).sha\n    state=hub.state()\n    if not state or not state.get(\'working\'):raise RuntimeError(\'Saved working checkpoint is missing.\')\n    if expected_updates is not None and state.get(\'updates\')!=expected_updates:\n        raise RuntimeError(\'Checkpoint step differs from EXPECTED_UPDATES; verify the revision before testing.\')\n    model,payload=deserialize(hub.read_entry(state[\'working\']))\n    del payload;gc.collect()\n    torch.set_num_threads(2);model.to(\'cuda\').eval()\n    print(json.dumps({\'decodeProbeCheckpoint\':state[\'updates\'],\'revision\':hub.revision,\n        \'weightObjectSha256\':state[\'working\'][\'sha256\'],\'GPU\':torch.cuda.get_device_name(0),\n        \'precision\':\'float32\',\'trainingStarted\':False}),flush=True)\n    report=compare(model,max_bytes)\n    report.update(format=\'nebula-v18-decoding-comparison-1\',checkpointUpdates=state[\'updates\'],\n        checkpointRevision=hub.revision,weightObjectSha256=state[\'working\'][\'sha256\'],\n        generatedAt=datetime.now(timezone.utc).isoformat(),maxGeneratedBytes=max_bytes,\n        notice=\'Synthetic inference probes only. No weight/optimizer updates. No byte repetition penalty applied.\')\n    if save_report:\n        name=\'decode-\'+datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%SZ\')+\'-\'+uuid.uuid4().hex[:8]+\'.json\'\n        path=\'experiments/\'+run_name+\'/decode-reports/\'+name\n        commit=hub.api.create_commit(repo_id=repo,operations=[CommitOperationAdd(\n            path_in_repo=path,path_or_fileobj=json.dumps(report,indent=2).encode())],\n            commit_message=\'Save inference-only decoding comparison\')\n        report[\'savedReportUrl\']=\'https://huggingface.co/\'+repo+\'/blob/\'+commit.oid+\'/\'+path\n        print(json.dumps({\'decodeReportSaved\':True,\'url\':report[\'savedReportUrl\'],\n                          \'trainingManifestModified\':False}),flush=True)\n    del model;gc.collect();torch.cuda.empty_cache();hub.session.close()\n    return report\n', 'v18_foundation_control.py': '"""Audit actual saved raw batches and run a bounded story foundation experiment."""\nimport copy\nimport gc\nimport gzip\nimport json\nimport math\nimport os\nimport random\nimport time\nimport unicodedata\nfrom pathlib import Path\n\nimport numpy as np\nimport requests\nimport torch\nimport torch.nn.functional as F\nfrom nebula import BOS,EOS\nfrom v18_stream import Hub,RemotePool,deserialize,serialize,sha,digest,emit,PROBES\nfrom v18_decode_probe import compare\n\nSOURCE_REVISION=\'e167fff3bb27c36c25460fcd4c61915a5b7016ed\'\nDEFAULT={\'sourceRepo\':\'iiidevv/nebula-raw-training\',\'sourceRun\':\'nebula-v18-raw-1gb\',\n         \'sourceRevision\':SOURCE_REVISION,\'expectedUpdates\':10102,\'updates\':1000,\n         \'trainUtf8Bytes\':8_000_000,\'validationUtf8Bytes\':500_000,\'seed\':19021,\n         \'saveEvery\':100,\'maxSessionHours\':2}\n\n\ndef code_signature():\n    root=Path(__file__).parent\n    return sha(\'\'.join((root/n).read_text(encoding=\'utf-8\') for n in\n               [\'v18_foundation_control.py\',\'nebula.py\',\'v18_stream.py\',\'v18_decode_probe.py\']).encode())\n\n\ndef read_source(repo,run,revision,token):\n    from huggingface_hub import HfApi\n    hub=object.__new__(Hub);hub.repo=repo;hub.prefix=\'experiments/\'+run+\'/\'\n    hub.api=HfApi(token=token);hub.token=token;hub.session=requests.Session()\n    hub.revision=hub.api.model_info(repo,revision=revision).sha\n    state=hub.state()\n    if not state or not state.get(\'working\') or not state.get(\'corpusComplete\'):\n        raise ValueError(\'The pinned source needs a working checkpoint and completed corpus.\')\n    return hub,state\n\n\ndef load_index(hub,state):\n    records=[];seen=set();counts={};sizes={};split_counts={\'train\':0,\'validation\':0}\n    spans={};previous={}\n    for entry in state[\'corpus\'][\'index\']:\n        for line in gzip.decompress(hub.read_entry(entry)).splitlines():\n            record=json.loads(line);identifier=record[\'hash\'];split=record[\'split\']\n            if split not in split_counts:raise ValueError(\'Invalid document split\')\n            if identifier in seen:raise ValueError(\'Duplicate normalized document hash or split overlap\')\n            if (\'validation\' if int(identifier[:8],16)%100<state[\'config\'][\'validationPercent\'] else \'train\')!=split:\n                raise ValueError(\'Document hash split differs from the training preparation rule\')\n            shard=record[\'shard\'];start=record[\'startToken\'];end=record[\'endToken\']\n            if not 0<=shard<len(state[\'corpus\'][split]):raise ValueError(\'Invalid shard index\')\n            if not 0<=start<end<=state[\'corpus\'][split][shard][\'bytes\']//2:\n                raise ValueError(\'Document range outside its saved shard\')\n            if end-start!=record[\'utf8Bytes\']+2:raise ValueError(\'Document length differs from byte count plus markers\')\n            key=(split,shard)\n            if previous.get(key,0)!=start:raise ValueError(\'Gap or overlap in indexed packed documents\')\n            previous[key]=end;spans[key]=end\n            seen.add(identifier);source=record[\'source\'];counts[source]=counts.get(source,0)+1\n            sizes[source]=sizes.get(source,0)+record[\'utf8Bytes\'];split_counts[split]+=1;records.append(record)\n        emit(controlAuditIndexDocuments=len(records))\n    for split in split_counts:\n        for i,entry in enumerate(state[\'corpus\'][split]):\n            if spans.get((split,i),0)!=entry[\'bytes\']//2:raise ValueError(\'Index does not cover the entire saved shard\')\n    if sum(sizes.values())!=state[\'uniqueTextBytes\'] or sizes!=state[\'sourceBytes\']:\n        raise ValueError(\'Indexed source totals differ from the saved corpus manifest\')\n    if len(records)!=state[\'documents\']:raise ValueError(\'Indexed document count differs from manifest\')\n    return records,{\'documents\':len(records),\'sourceDocuments\':counts,\'sourceUtf8Bytes\':sizes,\n        \'splitDocuments\':split_counts,\'uniqueNormalizedHashes\':len(seen),\n        \'indexCoverageComplete\':True,\'trainValidationHashesDisjoint\':True,\n        \'scope\':\'All index metadata checked; payload integrity/UTF-8/content checked on samples and selected story documents.\'}\n\n\ndef document(pool,record):\n    # Pool entries must retain original order: constructor filters undersized shards.\n    values=pool.data(record[\'shard\'])[record[\'startToken\']:record[\'endToken\']]\n    if len(values)!=record[\'utf8Bytes\']+2 or values[0]!=BOS or values[-1]!=EOS:\n        raise ValueError(\'Saved document boundary/length mismatch\')\n    interior=values[1:-1]\n    if np.any(interior>255):raise ValueError(\'Unexpected role/special token inside raw document\')\n    content=interior.astype(np.uint8).tobytes();text=content.decode(\'utf-8\',errors=\'strict\')\n    identifier=sha(\' \'.join(unicodedata.normalize(\'NFKC\',text).casefold().split()).encode())\n    if identifier!=record[\'hash\']:raise ValueError(\'Payload document hash differs from source index\')\n    return values,text\n\n\ndef audit_payloads(hub,state,records,context):\n    # No shard filtering here: an end shard can be shorter than context.\n    pools={}\n    for split in [\'train\',\'validation\']:\n        pool=object.__new__(RemotePool);pool.hub=hub;pool.entries=state[\'corpus\'][split]\n        from collections import OrderedDict\n        pool.cache=OrderedDict();pools[split]=pool\n    samples=[]\n    for source in sorted(state[\'sourceBytes\']):\n        for split in [\'train\',\'validation\']:\n            rows=sorted((r for r in records if r[\'source\']==source and r[\'split\']==split),key=lambda r:r[\'hash\'])[:4]\n            for row in rows:\n                _,text=document(pools[split],row)\n                samples.append({\'source\':source,\'split\':split,\'hash\':row[\'hash\'],\'utf8Bytes\':row[\'utf8Bytes\'],\n                                \'textBeginning\':text[:500]})\n    # Reconstruct the real V18 random batch at the saved optimizer update.\n    original=RemotePool(hub,state[\'corpus\'][\'train\'],context,state[\'config\'][\'shardHoldSteps\'])\n    step=state[\'updates\'];seed=state[\'config\'][\'seed\'];batch=state[\'config\'][\'batch\']\n    chosen=random.Random(seed+step//original.hold).choices(range(len(original.entries)),weights=original.weights,k=1)[0]\n    data=original.data(chosen);rng=random.Random(seed+step*7919)\n    pairs=original.batch(step,batch,seed);windows=[]\n    for x,y in pairs:\n        start=rng.randrange(len(data)-context)\n        expected=torch.tensor(np.array(data[start:start+context+1],dtype=np.int64))\n        if not torch.equal(x,expected[:-1]) or not torch.equal(y,expected[1:]) or not torch.equal(x[1:],y[:-1]):\n            raise ValueError(\'Actual V18 input/target alignment failed\')\n        if int(x.min())<0 or int(y.min())<0 or int(x.max())>EOS or int(y.max())>EOS:\n            raise ValueError(\'Unexpected token in raw training batch\')\n        windows.append({\'startToken\':start,\'inputLength\':len(x),\'targetLength\':len(y),\n                        \'bosTargets\':int((y==BOS).sum()),\'eosTargets\':int((y==EOS).sum()),\n                        \'decodedBeginning\':bytes(int(v) for v in x[:300] if v<256).decode(\'utf-8\',errors=\'replace\')})\n    result={\'payloadSamplesChecked\':len(samples),\'samples\':samples,\'actualV18Batch\':{\n        \'reconstructedUpdate\':step,\'batch\':batch,\'context\':context,\'targetShiftCorrect\':True,\'windows\':windows},\n        \'notice\':\'A window beginning mid-word or cutting a UTF-8 character is expected in byte-level packed training. Samples are not a full semantic/PII audit.\'}\n    del original,data\n    for pool in pools.values():pool.cache.clear()\n    gc.collect()\n    return pools,result\n\n\ndef select_stories(pools,records,train_limit,valid_limit):\n    arrays={};selected={};prefixes=[]\n    for split,limit in [(\'train\',train_limit),(\'validation\',valid_limit)]:\n        packed=bytearray();hashes=[];total=0\n        rows=sorted((r for r in records if r[\'source\']==\'stories\' and r[\'split\']==split),key=lambda r:r[\'hash\'])\n        # Read selected records in shard order to avoid re-downloading the same shard for each hash.\n        chosen=[]\n        for r in rows:\n            if total+r[\'utf8Bytes\']>limit:continue\n            chosen.append(r);total+=r[\'utf8Bytes\']\n            if total>=limit*.995:break\n        for r in sorted(chosen,key=lambda r:(r[\'shard\'],r[\'startToken\'])):\n            values,text=document(pools[split],r);packed.extend(values.tobytes());hashes.append(r[\'hash\'])\n            if split==\'validation\' and len(prefixes)<2:\n                prefix=text[:85].rsplit(\' \',1)[0]\n                if prefix:prefixes.append(prefix)\n        if not packed:raise ValueError(\'No indexed TinyStories documents available in \'+split)\n        arrays[split]=np.frombuffer(packed,dtype=\'<u2\')\n        selected[split]={\'documents\':len(hashes),\'utf8Bytes\':total,\'packedSha256\':sha(packed),\'hashes\':hashes}\n        pools[split].cache.clear()\n    if set(selected[\'train\'][\'hashes\'])&set(selected[\'validation\'][\'hashes\']):raise ValueError(\'Story split leakage\')\n    return arrays,selected,prefixes\n\n\ndef pairs_from_array(data,context,batch,seed):\n    if len(data)<=context:raise ValueError(\'Story pool too short for full context\')\n    rng=random.Random(seed);pairs=[]\n    for _ in range(batch):\n        start=rng.randrange(len(data)-context)\n        values=torch.tensor(np.array(data[start:start+context+1],dtype=np.int64))\n        pairs.append((values[:-1],values[1:]))\n    return pairs\n\n\n@torch.inference_mode()\ndef heldout(model,pairs):\n    model.eval();device=next(model.parameters()).device;total=0.;targets=0\n    for i,(x,y) in enumerate(pairs):\n        loss=F.cross_entropy(model(x[None].to(device)).float().flatten(0,1),y.to(device),reduction=\'sum\')\n        if not torch.isfinite(loss):raise ValueError(\'Non-finite held-out loss\')\n        total+=float(loss);targets+=len(y)\n        if (i+1)%8==0:emit(controlHeldoutBlocks=i+1,total=len(pairs))\n    return {\'loss\':total/targets,\'targets\':targets}\n\n\n@torch.inference_mode()\ndef audit_forward(model,pair):\n    model.eval();device=next(model.parameters()).device\n    x,y=(v[:96].to(device) for v in pair)\n    before=model(x[None]).float()\n    changed=x.clone();cut=len(x)//2;changed[cut:]=(changed[cut:]+17)%256\n    after=model(changed[None]).float()\n    if not torch.isfinite(before).all() or not torch.isfinite(after).all():raise ValueError(\'Non-finite source logits\')\n    delta=float((before[:,:cut]-after[:,:cut]).abs().max())\n    if not torch.allclose(before[:,:cut],after[:,:cut],rtol=1e-4,atol=1e-4):\n        raise ValueError(\'Future-token changes affected earlier logits on a real saved batch\')\n    batched=F.cross_entropy(before.flatten(0,1),y)\n    reference=-torch.log_softmax(before[0],-1)[torch.arange(len(y),device=device),y].mean()\n    if not torch.allclose(batched,reference,rtol=1e-5,atol=1e-5):raise ValueError(\'Reference causal loss mismatch\')\n    return {\'futureChangeMaxPastLogitDelta\':delta,\'causalPrefixInvariant\':True,\n            \'manualNextTokenLossMatches\':True,\'sampleLoss\':float(batched),\'positions\':len(y)}\n\n\ndef run_control(output_run=\'nebula-v18-foundation-control\',config=None,device=\'cuda\',clock=None):\n    clock=time.monotonic if clock is None else clock\n    config=copy.deepcopy(DEFAULT if config is None else config)\n    if set(config)!=set(DEFAULT) or not 1<=config[\'updates\']<=5000 or not 0<config[\'maxSessionHours\']<=8:\n        raise ValueError(\'Invalid controlled experiment configuration\')\n    if output_run==config[\'sourceRun\']:raise ValueError(\'Use a different run; the source must be preserved\')\n    if device==\'cuda\' and not torch.cuda.is_available():raise RuntimeError(\'Enable a Kaggle NVIDIA GPU\')\n    token=os.environ.get(\'NEBULA_HF_TOKEN\') or os.environ.get(\'HF_TOKEN\')\n    if not token:raise RuntimeError(\'Enable Kaggle secret NEBULA_HF_TOKEN\')\n    source,source_state=read_source(config[\'sourceRepo\'],config[\'sourceRun\'],config[\'sourceRevision\'],token)\n    if source_state[\'updates\']!=config[\'expectedUpdates\']:raise ValueError(\'Pinned source step mismatch\')\n    output=Hub(config[\'sourceRepo\'],token,output_run);state=output.state();signature=digest({\'config\':config,\'code\':code_signature()})\n    if state and state.get(\'signature\')!=signature:raise ValueError(\'Keep the original experiment settings/code to resume; otherwise choose a NEW output run\')\n    if state and state.get(\'complete\'):\n        emit(controlAlreadyComplete=True,updates=state[\'updates\'],publicWebsiteUpdated=False)\n        return json.loads(output.read_entry(state[\'finalReport\']))\n    if state is None:\n        state={\'format\':\'nebula-v18-foundation-control-1\',\'signature\':signature,\'config\':config,\n            \'sourceRevision\':source.revision,\'sourceWeightSha256\':source_state[\'working\'][\'sha256\'],\n            \'sourceUpdates\':source_state[\'updates\'],\'updates\':0,\'complete\':False,\'qualityApproved\':False,\'publicWebsiteUpdated\':False}\n    records,index_audit=load_index(source,source_state)\n    model,payload=deserialize(output.read_entry(state[\'working\']) if state.get(\'working\') else source.read_entry(source_state[\'working\']))\n    context=model.config.context\n    pools,payload_audit=audit_payloads(source,source_state,records,context)\n    arrays,selection,extra_prefixes=select_stories(pools,records,config[\'trainUtf8Bytes\'],config[\'validationUtf8Bytes\'])\n    selection_hash=digest(selection)\n    if state.get(\'selectionHash\') and state[\'selectionHash\']!=selection_hash:raise ValueError(\'Reconstructed story selection changed\')\n    model.to(device);torch.set_num_threads(2)\n    if not state.get(\'audit\'):\n        actual_pair=RemotePool(source,source_state[\'corpus\'][\'train\'],context,source_state[\'config\'][\'shardHoldSteps\']).batch(\n            source_state[\'updates\'],1,source_state[\'config\'][\'seed\'])[0]\n        forward_audit=audit_forward(model,actual_pair)\n    else:forward_audit={\'notice\':\'Original source forward audit already saved with this experiment.\'}\n    audit={\'index\':index_audit,\'payload\':payload_audit,\'storySelection\':selection,\'forward\':forward_audit,\n           \'passed\':True,\'humanVerified\':False,\'scope\':\'Packing/alignment/integrity checks, not proof of semantic correctness or full model architecture.\'}\n    emit(controlAuditSummary=index_audit,forwardAudit=forward_audit,\n         actualBatchChecks={k:v for k,v in payload_audit[\'actualV18Batch\'].items() if k!=\'windows\'})\n    seen_samples=set()\n    for sample in payload_audit[\'samples\']:\n        key=(sample[\'source\'],sample[\'split\'])\n        if key not in seen_samples:\n            emit(controlPublicDocumentSample=sample);seen_samples.add(key)\n    for window in payload_audit[\'actualV18Batch\'][\'windows\'][:2]:emit(controlActualWindow=window)\n    if not state.get(\'audit\'):\n        state=output.publish({**state,\'selectionHash\':selection_hash},{\'audit\':(\'audit.json\',json.dumps(audit,indent=2).encode())})\n    emit(controlAuditPassed=True,storyTrainBytes=selection[\'train\'][\'utf8Bytes\'],storyValidationBytes=selection[\'validation\'][\'utf8Bytes\'])\n    del records,pools,audit,payload_audit;gc.collect()\n    amp=device==\'cuda\';native_bf16=amp and torch.cuda.get_device_capability(0)[0]>=8 and torch.cuda.is_bf16_supported()\n    dtype=torch.bfloat16 if native_bf16 else torch.float16\n    scaler=torch.amp.GradScaler(\'cuda\',enabled=amp and not native_bf16,init_scale=1024)\n    optimizer=torch.optim.AdamW(model.parameters(),lr=source_state[\'config\'][\'learningRate\'],weight_decay=.01)\n    training=payload[\'trainingState\'];optimizer.load_state_dict(training[\'optimizer\'])\n    if state.get(\'working\'):\n        if training.get(\'controlSignature\')!=signature:raise ValueError(\'Controlled checkpoint signature mismatch\')\n        if training[\'updates\']!=state[\'updates\']:raise ValueError(\'Controlled checkpoint step mismatch\')\n    if scaler.is_enabled() and training.get(\'scaler\'):scaler.load_state_dict(training[\'scaler\'])\n    del payload,training;gc.collect()\n    mixed=RemotePool(source,source_state[\'corpus\'][\'validation\'],context,1)\n    mixed_pairs=[mixed.batch(i,1,880019)[0] for i in range(16)]\n    story_pairs=pairs_from_array(arrays[\'validation\'],context,16,config[\'seed\']+1)\n    del mixed;gc.collect()\n    modes=[dict(name=\'greedy\',temperature=0.,top_p=1.,seeds=[0]),\n           dict(name=\'sample-0.7\',temperature=.7,top_p=.9,seeds=[17,43])]\n    def evaluate(label):\n        emit(controlEvaluation=label)\n        report={\'stage\':label,\'additionalUpdates\':state[\'updates\'],\n                \'heldoutMixed\':heldout(model,mixed_pairs),\'heldoutStories\':heldout(model,story_pairs),\n                **compare(model,128,modes=modes,raw=PROBES+extra_prefixes)}\n        # compare\'s generic inference-only flag refers to the probe, not this whole experiment.\n        report[\'trainingStarted\']=state[\'updates\']>0\n        return report\n    if not state.get(\'baselineReport\'):\n        baseline=evaluate(\'baseline-10102\')\n        state=output.publish(state,{\'baselineReport\':(\'baseline.json\',json.dumps(baseline,indent=2).encode())})\n    started=clock();bad=0;last_saved=state[\'updates\'];interrupted=False\n    source_config=source_state[\'config\'];batch=source_config[\'batch\'];microbatch=source_config[\'microbatch\']\n    emit(controlTraining=True,startingStep=source_state[\'updates\'],additionalUpdateBudget=config[\'updates\'],\n         optimizerRestored=True,precision=str(dtype) if amp else \'float32\',trainingGPUs=1,\n         change=\'Train only on a deterministic public TinyStories subset; retain model, batch and original learning-rate schedule.\',\n         limitedChange=\'T4 uses native FP16 AMP instead of emulated BF16 when inherited source used it.\',publicWebsiteUpdated=False)\n    def save():\n        nonlocal state,last_saved\n        stream=serialize(model,{\'experimental\':True,\'qualityApproved\':False,\'sourceUpdates\':source_state[\'updates\'],\n                     \'additionalUpdates\':state[\'updates\']},\n            {\'controlSignature\':signature,\'optimizer\':optimizer.state_dict(),\'scaler\':scaler.state_dict(),\'updates\':state[\'updates\']})\n        try:state=output.publish(state,{\'working\':(\'working.pt\',stream.getbuffer())})\n        finally:stream.close()\n        last_saved=state[\'updates\']\n        emit(controlCheckpointConfirmed=True,additionalUpdates=last_saved,publicWebsiteUpdated=False)\n    try:\n        while state[\'updates\']<config[\'updates\'] and clock()-started<config[\'maxSessionHours\']*3600:\n            step=state[\'updates\'];absolute=source_state[\'updates\']+step\n            rate=source_config[\'learningRate\']*min(1.,(absolute+1)/source_config[\'warmup\'])*(.1+.9*.5*(1+math.cos(math.pi*absolute/source_config[\'steps\'])))\n            for group in optimizer.param_groups:group[\'lr\']=rate\n            model.train();optimizer.zero_grad(set_to_none=True);loss_value=0.\n            pairs=pairs_from_array(arrays[\'train\'],context,batch,config[\'seed\']+step*7919)\n            for offset in range(0,batch,microbatch):\n                micro=pairs[offset:offset+microbatch]\n                x=torch.stack([p[0] for p in micro]).to(device);y=torch.stack([p[1] for p in micro]).to(device)\n                with torch.autocast(\'cuda\',dtype=dtype,enabled=amp):\n                    loss=F.cross_entropy(model(x).flatten(0,1),y.flatten(),reduction=\'sum\')/(batch*context)\n                if not torch.isfinite(loss):raise ValueError(\'Non-finite loss; keep the last confirmed control checkpoint\')\n                loss_value+=float(loss.detach());scaler.scale(loss).backward()\n            scaler.unscale_(optimizer);norm=torch.nn.utils.clip_grad_norm_(model.parameters(),1.)\n            if not torch.isfinite(norm):\n                bad+=1\n                if not scaler.is_enabled() or bad>12:raise ValueError(\'Non-finite gradients\')\n                scaler.update(new_scale=scaler.get_scale()/2);continue\n            scaler.step(optimizer);scaler.update();bad=0;state[\'updates\']+=1\n            if state[\'updates\']%25==0:\n                emit(controlUpdate=state[\'updates\'],budget=config[\'updates\'],loss=round(loss_value,4),\n                     learningRate=rate,trainingBytePositions=state[\'updates\']*batch*context)\n            if state[\'updates\']%config[\'saveEvery\']==0:save()\n    except KeyboardInterrupt:interrupted=True\n    if state[\'updates\']!=last_saved:save()\n    if interrupted or state[\'updates\']<config[\'updates\']:\n        emit(controlPaused=True,updatesSaved=last_saved,runAgainWithSameSettings=True)\n        return state\n    final=evaluate(\'story-foundation-final\')\n    baseline=json.loads(output.read_entry(state[\'baselineReport\']))\n    report={\'format\':\'nebula-foundation-control-comparison-1\',\'sourceRevision\':source.revision,\n            \'sourceWeightSha256\':source_state[\'working\'][\'sha256\'],\'baseline\':baseline,\'final\':final,\n            \'additionalUpdates\':state[\'updates\'],\'additionalTrainingPositions\':state[\'updates\']*batch*context,\n            \'qualityApproved\':False,\'humanVerified\':False,\'publicWebsiteUpdated\':False,\n            \'interpretation\':\'Compare actual completions and both held-out losses. Repetition reductions alone do not establish coherence. No automatic promotion or source replacement.\'}\n    state=output.publish({**state,\'complete\':True},{\'finalReport\':(\'comparison.json\',json.dumps(report,indent=2).encode())})\n    emit(controlFinished=True,updatesSaved=state[\'updates\'],sourceWeightsPreserved=True,publicWebsiteUpdated=False,\n         reportParts=[p[\'path\'] for p in state[\'finalReport\'][\'parts\']])\n    del optimizer,model;gc.collect()\n    if amp:torch.cuda.empty_cache()\n    return report\n', 'v19_tutor_data.py': '"""Original, deterministic tutor lessons; benchmark prompts never enter practice."""\nimport hashlib\nimport json\nimport random\nimport re\nfrom collections import Counter\n\nimport torch\nfrom nebula import encode,PAD\n\nSYSTEM={\'role\':\'system\',\'content\':\'You are Nebula AI.\'}\nNAMES=[\'Ada\',\'Ben\',\'Cleo\',\'Dario\',\'Evan\',\'Faye\',\'Hugo\',\'Mina\',\'Noah\',\'Omar\',\'Rosa\',\'Tess\']\nCOLORS=[\'red\',\'blue\',\'green\',\'white\',\'black\',\'yellow\',\'pink\',\'gold\']\nNEW_NAMES=[\'Iris\',\'Juno\',\'Kai\',\'Lina\',\'Talia\',\'Marta\']\nNEW_COLORS=[\'purple\',\'orange\',\'silver\',\'brown\']\nMEANINGS=[(\'cold\',\'having a low temperature\',\'The water is cold.\'),\n (\'hot\',\'having a high temperature\',\'The soup is hot.\'),\n (\'empty\',\'containing nothing\',\'The cup is empty.\'),(\'full\',\'filled to capacity\',\'The cup is full.\'),\n (\'quiet\',\'making little noise\',\'The room is quiet.\'),(\'loud\',\'making a lot of noise\',\'The music is loud.\'),\n (\'wet\',\'covered with water\',\'The towel is wet.\'),(\'dry\',\'not wet\',\'The towel is dry.\'),\n (\'before\',\'earlier than\',\'Breakfast comes before lunch.\'),(\'after\',\'later than\',\'Dinner comes after lunch.\'),\n (\'above\',\'higher than\',\'The lamp is above the table.\'),(\'below\',\'lower than\',\'The floor is below the table.\'),\n (\'near\',\'not far away\',\'The chair is near the desk.\'),(\'far\',\'a long distance away\',\'The town is far away.\'),\n (\'borrow\',\'use something temporarily and return it\',\'I borrow a book and return it.\'),\n (\'return\',\'give something back\',\'I return the book to its owner.\')]\nFACTS=[(\'Why does ice melt?\',\'Ice melts when it absorbs enough heat to change from solid water to liquid water.\'),\n (\'What is ice?\',\'Ice is water in its solid form.\'),\n (\'What do roots do for a plant?\',\'Roots take in water and help hold the plant in the ground.\'),\n (\'Why do we use an umbrella?\',\'An umbrella can help keep rain off us.\'),\n (\'What does a thermometer measure?\',\'A thermometer measures temperature.\'),\n (\'What is a sentence?\',\'A sentence is a group of words that expresses a complete thought.\'),\n (\'What is a question?\',\'A question asks for information.\'),\n (\'What is a noun?\',\'A noun names a person, place, thing, or idea.\'),\n (\'What is a verb?\',\'A verb can describe an action or a state.\'),\n (\'What does a map show?\',\'A map shows where places are and how they are arranged.\'),\n (\'Why should we check an answer?\',\'Checking can help us find and correct mistakes.\'),\n (\'What should you do if a question is unclear?\',\'Ask a question to find out what the person means.\')]\nCHATS=[(\'Hello!\',\'Hello! How can I help?\'),(\'Hi there.\',\'Hi! What would you like to work on?\'),\n (\'What is your name?\',\'I am Nebula AI.\'),(\'Who are you?\',\'I am Nebula AI, an AI assistant.\'),\n (\'Can you help me write an email?\',\'Yes. Who is the email for, and what would you like it to say?\'),\n (\'Which should I choose?\',\'What options are you choosing between, and what matters most to you?\'),\n (\'Can you explain this?\',\'What would you like me to explain?\'),\n (\'Help me plan something.\',\'What are you planning, and when does it need to happen?\'),\n (\'Thank you for helping.\',\'You are welcome!\'),(\'I made a mistake.\',\'We can work through it. What happened?\'),\n (\'I do not understand the question.\',\'Share the question, and we can break it into smaller parts.\'),\n (\'Tell me a short story.\',\'A child found a lost dog near the park. She checked its tag and helped it get home.\'),\n (\'Write a short thank-you note to a neighbor.\',\'Thank you for your help. I appreciate your kindness and am glad we are neighbors.\'),\n (\'Can you read my mind?\',\'No. I only know what is available in the conversation and any information provided to me.\'),\n (\'What is inside my closed box?\',\'I do not know. Can you describe what is inside it?\'),\n (\'Can you be certain without enough information?\',\'No. I should explain what is uncertain and ask for the missing information.\')]\n\n\ndef pair(question,answer):return [dict(SYSTEM),{\'role\':\'user\',\'content\':question},{\'role\':\'assistant\',\'content\':answer}]\ndef key(messages):return hashlib.sha256(json.dumps(messages,sort_keys=True).encode()).hexdigest()\ndef norm(text):return \' \'.join(text.casefold().split())\n\n\ndef benchmarks(seed=190221):\n    rng=random.Random(seed);cases=[]\n    def add(axis,messages,expected):cases.append({\'axis\':axis,\'messages\':messages,\'expected\':str(expected)})\n    for i in range(12):\n        name=NEW_NAMES[i%6];color=NEW_COLORS[i%4]\n        ask=\'name\' if i%2==0 else \'color\'\n        fields=[f\'Name = {name}\',f\'Scarf color = {color}\',f\'Number = {21+i}\'];rng.shuffle(fields)\n        q=\'; \'.join(fields)+f\'. Read this record and return the {ask} alone.\'\n        add(\'new-values\',pair(q,\'\')[:-1],name if ask==\'name\' else color)\n    for i in range(12):\n        name=NAMES[i];color=COLORS[(i*3)%len(COLORS)];ask=\'name\' if i%2==0 else \'color\'\n        q=f\'The person is {name}; their scarf is {color}. Extract just the \'+(\'person\\\'s name.\' if ask==\'name\' else \'scarf color.\')\n        add(\'new-wording\',pair(q,\'\')[:-1],name if ask==\'name\' else color)\n    for i in range(16):\n        name=NEW_NAMES[i%6];old=COLORS[i%8];new=NEW_COLORS[i%4];ask_name=i%4==0\n        messages=pair(f\'My name is {name}. My scarf is {old}.\',f\'Your name is {name}, and your scarf is {old}.\')\n        messages.extend([{\'role\':\'user\',\'content\':f\'Update: my scarf is now {new}, not {old}.\'},\n                         {\'role\':\'assistant\',\'content\':\'Understood. I will use the updated color.\'},\n                         {\'role\':\'user\',\'content\':\'Give only \'+(\'my name.\' if ask_name else \'the current scarf color.\')}])\n        add(\'updated-multi-turn\',messages,name if ask_name else new)\n    for i in range(8):\n        a=NEW_NAMES[i%6];b=NEW_COLORS[i%4];c=str(31+i)\n        add(\'instruction\',pair(f\'Items in order: {a}, {b}, {c}. Return only the final item.\',\'\')[:-1],c)\n    for i in range(8):\n        a=17+i;b=3+i\n        add(\'arithmetic\',pair(f\'Calculate {a} + {b}. Output the number alone.\',\'\')[:-1],a+b)\n    for word,definition,_ in MEANINGS[:8]:\n        add(\'meaning\',pair(f\'Which word means "{definition}"? Return one word.\',\'\')[:-1],word)\n    return cases\n\n\ndef lessons(seed=190221,dynamic=1600):\n    rng=random.Random(seed);rows=[];seen=set();tests=benchmarks(seed)\n    banned={norm(m[\'content\']) for c in tests for m in c[\'messages\'] if m[\'role\']==\'user\'}\n    def add(family,messages):\n        identifier=key(messages)\n        if identifier in seen or any(norm(m[\'content\']) in banned for m in messages if m[\'role\']==\'user\'):return\n        seen.add(identifier);rows.append({\'family\':family,\'messages\':messages,\'id\':identifier,\'origin\':\'original-synthetic-tutor\'})\n    for word,definition,sentence in MEANINGS:\n        for question,answer in [(f\'What does {word} mean?\',f\'{word.capitalize()} means {definition}.\'),\n          (f\'Explain the word "{word}" and use it in a sentence.\',f\'It means {definition}. For example: {sentence}\'),\n          (f\'Give a sentence using {word}.\',sentence),\n          (f\'Explain this sentence: {sentence}\',f\'The word "{word}" means {definition} here.\')]:\n            add(\'meaning\',pair(question,answer))\n    for question,answer in FACTS:\n        for lead in [\'\', \'Please explain simply: \', \'Help me understand: \']:add(\'explanation\',pair(lead+question,answer))\n    for question,answer in CHATS:\n        for lead in [\'\', \'Please help. \']:add(\'conversation\',pair(lead+question,answer))\n    for i in range(dynamic):\n        name=rng.choice(NAMES);color=rng.choice(COLORS);old=rng.choice([c for c in COLORS if c!=color]);number=rng.randrange(21)\n        family=i%5\n        if family==0:\n            ask=rng.choice([\'name\',\'color\',\'number\']);fields=[f\'Name: {name}\',f\'Color: {color}\',f\'Number: {number}\'];rng.shuffle(fields)\n            instruction=rng.choice([f\'Reply with only the {ask}.\',f\'What is the {ask}? Give only its value.\',f\'Find the {ask}. Answer with its value only.\'])\n            answer={\'name\':name,\'color\':color,\'number\':str(number)}[ask]\n            add(\'context\',pair(\'. \'.join(fields)+\'. \'+instruction,answer))\n        elif family==1:\n            messages=pair(f\'I am {name}. My scarf is {old}.\',f\'Hello, {name}. Your scarf is {old}.\')\n            ask_name=rng.randrange(4)==0\n            if rng.randrange(2):\n                messages.extend([{\'role\':\'user\',\'content\':f\'My scarf is {color} now.\'},\n                                 {\'role\':\'assistant\',\'content\':f\'Your current scarf color is {color}.\'}])\n            else:color=old\n            q=\'What is my name?\' if ask_name else \'What color is my scarf now?\'\n            answer=name if ask_name else color\n            if rng.randrange(2):q+=\' Reply with just the answer.\'\n            else:answer=f\'Your name is {name}.\' if ask_name else f\'Your scarf is {color}.\'\n            messages.extend([{\'role\':\'user\',\'content\':q},{\'role\':\'assistant\',\'content\':answer}]);add(\'multi-turn\',messages)\n        elif family==2:\n            kind=rng.randrange(3)\n            if kind==0:q=f\'Fix capitalization and punctuation: i like {color}\';a=f\'I like {color}.\'\n            elif kind==1:q=f\'Return only the last word in: {name} {color} {number}\';a=str(number)\n            else:q=f\'Put these on separate lines, without adding words: {name}, {color}, {number}\';a=f\'{name}\\n{color}\\n{number}\'\n            add(\'instruction\',pair(q,a))\n        elif family==3:\n            a=rng.randrange(16);b=rng.randrange(16);op=rng.choice([\'+\',\'-\'])\n            if op==\'-\':a,b=max(a,b),min(a,b)\n            answer=a+b if op==\'+\' else a-b\n            q=rng.choice([f\'What is {a} {op} {b}?\',f\'Work out {a} {op} {b}.\'])\n            if rng.randrange(2):q+=\' Give only the number.\';reply=str(answer)\n            else:reply=f\'{a} {op} {b} = {answer}.\'\n            add(\'arithmetic\',pair(q,reply))\n        else:\n            attribute=rng.choice([\'empty\',\'full\',\'wet\',\'dry\',\'hot\',\'cold\']);obj=rng.choice([\'cup\',\'bowl\',\'glass\'])\n            yes=rng.randrange(2)==0\n            properties={\'empty\':(\'contains nothing\',\'contains something\'),\'full\':(\'is filled to the top\',\'contains nothing\'),\n                        \'wet\':(\'has water on it\',\'has no water on it\'),\'dry\':(\'has no water on it\',\'has water on it\'),\n                        \'hot\':(\'has a high temperature\',\'has a low temperature\'),\'cold\':(\'has a low temperature\',\'has a high temperature\')}\n            statement=properties[attribute][0 if yes else 1]\n            q=f\'{name} has a {obj} that {statement}. Is the {obj} {attribute}?\'\n            if rng.randrange(2):q+=\' Reply yes or no.\';a=\'Yes\' if yes else \'No\'\n            else:a=(\'Yes\' if yes else \'No\')+f\'. The {obj} {statement}.\'\n            add(\'application\',pair(q,a))\n    # Practice is sampled equally by family, so a large template family cannot dominate updates.\n    counts=dict(Counter(r[\'family\'] for r in rows))\n    return rows,tests,{\'examples\':len(rows),\'familyExamples\':counts,\'practiceSampling\':\'equal probability per family\',\n        \'trainingNames\':NAMES,\'trainingColors\':COLORS,\'benchmarkOnlyNames\':NEW_NAMES,\'benchmarkOnlyColors\':NEW_COLORS,\n        \'heldoutScope\':\'New relative to generated tutor practice, not guaranteed unseen in all historical public pretraining.\',\n        \'benchmarkPromptsExcluded\':True,\'teacherCalls\':0}\n\n\ndef window(messages,context):\n    tokens,mask=encode(messages)\n    if len(tokens)>context+1:raise ValueError(\'Tutor example exceeds context; do not truncate lessons silently\')\n    x=torch.tensor(tokens[:-1]);y=torch.tensor([v if learn else -100 for v,learn in zip(tokens[1:],mask[1:])])\n    if not (y!=-100).any():raise ValueError(\'Tutor example has no assistant targets\')\n    return x,y\n\n\ndef collate(pairs):\n    length=max(len(x) for x,_ in pairs)\n    x=torch.full((len(pairs),length),PAD,dtype=torch.long);y=torch.full((len(pairs),length),-100,dtype=torch.long)\n    for i,(a,b) in enumerate(pairs):x[i,:len(a)]=a;y[i,:len(b)]=b\n    return x,y\n\n\ndef exact_grade(reply,expected):\n    # Strict full-answer match: mentioning the right word inside an incorrect story never passes.\n    return reply.strip().rstrip(\'.!?\').strip().casefold()==expected.strip().casefold()\n', 'v20_generalize_data.py': '"""Paired questions, varied copying and fact updates; original synthetic practice."""\nimport hashlib\nimport json\nimport random\nimport re\nfrom collections import Counter\nfrom v19_tutor_data import pair,key,norm,window,collate,exact_grade as relaxed_word_grade,MEANINGS,FACTS,lessons as old_lessons\n\n\ndef exact_grade(reply,expected,case_sensitive=False):\n    # Codes are identifiers: case, leading/trailing whitespace and punctuation must be preserved exactly.\n    return reply==expected if case_sensitive else relaxed_word_grade(reply,expected)\n\n\ndef inventory(seed):\n    rng=random.Random(seed)\n    syllables=[a+b for a in \'bdfgklmnprstvz\' for b in \'aeio\']\n    names=[]\n    while len(names)<340:\n        name=\'\'.join(rng.choices(syllables,k=rng.randrange(2,5))).capitalize()\n        if name not in names:names.append(name)\n    codes=[]\n    while len(codes)<1200:\n        code=\'\'.join(rng.choices(\'abcdefghjkmnpqrstuvwxyzABCDEFGHJKMNPQRSTUVWXYZ23456789\',k=rng.randrange(5,13)))\n        if code not in codes:codes.append(code)\n    return {\'names\':names[:300],\'heldNames\':names[300:], \'codes\':codes[:1100],\'heldCodes\':codes[1100:],\n      \'colors\':[\'red\',\'blue\',\'green\',\'white\',\'black\',\'yellow\',\'pink\',\'gold\',\'purple\',\'orange\',\'silver\',\'brown\',\n                \'cyan\',\'magenta\',\'lime\',\'olive\',\'plum\',\'amber\',\'tan\',\'peach\',\'mint\',\'violet\'],\n      \'heldColors\':[\'teal\',\'maroon\',\'beige\',\'coral\',\'navy\',\'ivory\'],\n      \'places\':[\'kitchen\',\'bedroom\',\'garden\',\'park\',\'school\',\'library\',\'hall\',\'office\',\'shed\',\'shop\'],\n      \'heldPlaces\':[\'attic\',\'garage\',\'balcony\',\'basement\']}\n\n\ndef profile(rng,values,held=False):\n    return {field:rng.choice(values[(\'held\'+keyname if held else keyname)]) for field,keyname in\n            [(\'name\',\'Names\' if held else \'names\'),(\'color\',\'Colors\' if held else \'colors\'),\n             (\'code\',\'Codes\' if held else \'codes\'),(\'place\',\'Places\' if held else \'places\')]}\n\n\ndef context(rng,p,style=None):\n    style=rng.randrange(3) if style is None else style\n    if style==0:\n        fields=[f\'Name: {p["name"]}\',f\'Color: {p["color"]}\',f\'Code: {p["code"]}\',f\'Place: {p["place"]}\']\n        rng.shuffle(fields);return \'; \'.join(fields)+\'.\'\n    if style==1:\n        sentences=[f\'{p["name"]} has a {p["color"]} scarf.\',f\'The scarf is in the {p["place"]}.\',f\'The label code is {p["code"]}.\']\n        rng.shuffle(sentences);return \' \'.join(sentences)\n    fields=[f\'PERSON={p["name"]}\',f\'SCARF={p["color"]}\',f\'TAG={p["code"]}\',f\'LOCATION={p["place"]}\']\n    rng.shuffle(fields);return \'Record: \'+\' | \'.join(fields)+\'.\'\n\n\nQUESTIONS={\n \'name\':[\'What is the name?\',\'Who is this person?\',\'Find the person\\\'s name.\',\'What name was provided?\',\n         \'Read out the name.\',\'Give the name from the record.\'],\n \'color\':[\'What color is the scarf?\',\'Find the scarf color.\',\'What is the color?\',\'Which color was given?\',\n          \'Give the color from the record.\',\'Read out the scarf color.\'],\n \'code\':[\'What is the code?\',\'Copy the label code.\',\'Find the tag code.\',\'What code was provided?\',\n         \'Give the code from the record.\',\'Read out the label.\'],\n \'place\':[\'Where is the scarf?\',\'What is the place?\',\'Find the location.\',\'Which place was given?\',\n          \'Give the location from the record.\',\'Read out the place.\']}\nTEST_QUESTIONS={\'name\':\'Identify the person by name.\',\'color\':\'Which scarf color is recorded?\',\n                \'code\':\'What exact tag text belongs to this record?\',\'place\':\'Identify the stored location.\'}\n\n\ndef held_math(a,b):return int(hashlib.sha256(f\'{a}+{b}\'.encode()).hexdigest()[:8],16)%5==0\n\n\ndef benchmarks(seed):\n    values=inventory(seed);rng=random.Random(seed+8501);cases=[]\n    def add(axis,messages,expected,field=\'value\'):\n        cases.append({\'axis\':axis,\'messages\':messages,\'expected\':str(expected),\'field\':field})\n    for i in range(12):\n        value=values[\'heldCodes\'][i]\n        add(\'verbatim-copy\',pair(f\'Text inside brackets: [{value}]. Output exactly the bracket contents.\',\'\')[:-1],value,\'code\')\n    for axis,held in [(\'unseen-values\',True),(\'unseen-wording\',False)]:\n        for i in range(12):\n            p=profile(rng,values,held);field=[\'name\',\'color\',\'code\',\'place\'][i%4]\n            q=context(rng,p)+\' \'+TEST_QUESTIONS[field]+\' Return the value and nothing else.\'\n            add(axis,pair(q,\'\')[:-1],p[field],field)\n    for i in range(12):\n        p=profile(rng,values,True);other=profile(rng,values)\n        while other[\'name\']==p[\'name\']:other=profile(rng,values)\n        text=f\'{p["name"]} has a {p["color"]} scarf. {other["name"]} has a {other["color"]} scarf.\'\n        add(\'entity-binding\',pair(text+f\' State only the scarf color belonging to {p["name"]}.\',\'\')[:-1],p[\'color\'],\'color\')\n    for i in range(12):\n        p=profile(rng,values,True);old=rng.choice(values[\'colors\']);field=\'name\' if i%3==0 else \'color\'\n        messages=pair(f\'My name is {p["name"]}. My scarf is {old}.\',\'I understand.\')\n        messages.extend([{\'role\':\'user\',\'content\':f\'Correction: the scarf is {p["color"]} now.\'},\n            {\'role\':\'assistant\',\'content\':\'I will use the updated information.\'},\n            {\'role\':\'user\',\'content\':(\'Identify me by name.\' if field==\'name\' else \'State the scarf color after my correction.\')+\' Return just the value.\'}])\n        add(\'updated-facts\',messages,p[field],field)\n    for i in range(12):\n        p=profile(rng,values,True);items=[p[\'name\'],p[\'color\'],p[\'code\']];index=i%3\n        add(\'instruction-selection\',pair(\'Sequence: \'+\' / \'.join(items)+f\'. Return item {index+1} alone.\',\'\')[:-1],items[index],[\'name\',\'color\',\'code\'][index])\n    pairs=[(a,b) for a in range(16,31) for b in range(1,31) if held_math(a,b)];rng.shuffle(pairs)\n    for a,b in pairs[:12]:add(\'arithmetic-combinations\',pair(f\'Compute the sum of {a} and {b}. Return a number only.\',\'\')[:-1],a+b,\'number\')\n    for word,definition,_ in MEANINGS[:12]:\n        add(\'meaning-transfer\',pair(f\'Name the word described by this definition: {definition}. One word only.\',\'\')[:-1],word)\n    return cases\n\n\ndef lessons(seed=200221,groups=1800):\n    rng=random.Random(seed);values=inventory(seed);cases=benchmarks(seed);rows=[];seen=set()\n    banned={norm(m[\'content\']) for c in cases for m in c[\'messages\'] if m[\'role\']==\'user\'}\n    def add(family,messages,origin=\'original-synthetic-v20\'):\n        identifier=key(messages)\n        if identifier in seen or any(norm(m[\'content\']) in banned for m in messages if m[\'role\']==\'user\'):return\n        seen.add(identifier);rows.append({\'family\':family,\'messages\':messages,\'id\':identifier,\'origin\':origin})\n    for i in range(groups):\n        p=profile(rng,values);text=context(rng,p);fields=rng.sample([\'name\',\'color\',\'code\',\'place\'],3)\n        # A single context supports several different answers. Query order varies independently of field order.\n        for field in fields:\n            add(\'binding\',pair(text+\' \'+rng.choice(QUESTIONS[field])+\' Answer with only the value.\',p[field]))\n        messages=pair(text+\' \'+rng.choice(QUESTIONS[fields[0]])+\' Only the value.\',p[fields[0]])\n        for field in fields[1:]:\n            messages.extend([{\'role\':\'user\',\'content\':rng.choice(QUESTIONS[field])+\' Only the value.\'},\n                             {\'role\':\'assistant\',\'content\':p[field]}])\n        add(\'binding\',messages)\n        value=rng.choice([p[\'name\'],p[\'code\'],p[\'color\'],str(rng.randrange(1000))])\n        for q in [f\'Copy exactly: {value}\',f\'Repeat only the text between quotes: "{value}"\']:\n            add(\'copy\',pair(q,value))\n        other=profile(rng,values)\n        while other[\'name\']==p[\'name\']:other=profile(rng,values)\n        text=f\'{p["name"]} has a {p["color"]} scarf. {other["name"]} has a {other["color"]} scarf.\'\n        for person in [p,other]:\n            add(\'entities\',pair(text+f\' What color is {person["name"]}\\\'s scarf? Reply only with the color.\',person[\'color\']))\n        new=rng.choice([c for c in values[\'colors\'] if c!=p[\'color\']])\n        messages=pair(f\'I am {p["name"]}. My scarf is {p["color"]}.\',\'Understood.\')\n        messages.extend([{\'role\':\'user\',\'content\':rng.choice([f\'My scarf is {new} now.\',f\'Change the scarf color to {new}.\',f\'The old color is wrong. It is {new}.\'])},\n                         {\'role\':\'assistant\',\'content\':f\'The current color is {new}.\'},\n                         {\'role\':\'user\',\'content\':\'What color is my scarf now? Answer only with the color.\'},\n                         {\'role\':\'assistant\',\'content\':new},\n                         {\'role\':\'user\',\'content\':\'What is my name? Reply with only the name.\'},\n                         {\'role\':\'assistant\',\'content\':p[\'name\']}])\n        add(\'updates\',messages)\n        items=[p[\'name\'],p[\'color\'],p[\'code\']]\n        for index in range(3):\n            q=rng.choice([f\'Return only item {index+1} from: \',f\'Give the \'+[\'first\',\'second\',\'third\'][index]+\' value only: \'])\n            add(\'instructions\',pair(q+\', \'.join(items),items[index]))\n        a=rng.randrange(31);b=rng.randrange(31)\n        if not held_math(a,b):\n            q=rng.choice([f\'What is {a} + {b}?\',f\'Add {a} and {b}.\',f\'Find the sum: {a} + {b}.\'])+\' Give only the number.\'\n            messages=pair(q,str(a+b))\n            messages.extend([{\'role\':\'user\',\'content\':\'What was the first number? Give only that number.\'},\n                             {\'role\':\'assistant\',\'content\':str(a)},\n                             {\'role\':\'user\',\'content\':\'What was the second number? Give only that number.\'},\n                             {\'role\':\'assistant\',\'content\':str(b)}])\n            add(\'arithmetic\',messages)\n    for word,definition,sentence in MEANINGS:\n        for q,a in [(f\'What word means {definition}? Answer with one word.\',word),\n                    (f\'Give only the word for this meaning: {definition}.\',word),\n                    (f\'What does {word} mean?\',f\'{word.capitalize()} means {definition}.\'),\n                    (f\'Use {word} in a sentence.\',sentence)]:add(\'meaning\',pair(q,a))\n    for q,a in FACTS:\n        for lead in [\'\', \'Explain simply: \', \'Please help me understand: \']:add(\'meaning\',pair(lead+q,a))\n    favors=[\'watering my plants\',\'the book\',\'helping me move\',\'checking on my cat\',\'your advice\',\'helping with my homework\']\n    for name in values[\'names\'][:100]:\n        favor=rng.choice(favors)\n        messages=pair(f\'Write a short note to {name} thanking them for {favor}.\',f\'{name}, thank you for {favor}. I appreciate it.\')\n        messages.extend([{\'role\':\'user\',\'content\':\'Make the note shorter.\'},\n                         {\'role\':\'assistant\',\'content\':f\'Thank you for {favor}, {name}.\'}]);add(\'conversation\',messages)\n        add(\'conversation\',pair(f\'I am choosing between {name}\\\'s plan and another plan. What information do you need?\',\n                              \'What are the plans, and what matters most to you?\'))\n    # Retain useful V19 responses, but cap replay at 20% of tutor examples during sampling.\n    old,_,_=old_lessons(190221)\n    for row in old:add(\'replay/\'+row[\'family\'],row[\'messages\'],\'v19-practice-replay\')\n    return rows,cases,{\'examples\':len(rows),\'familyExamples\':dict(Counter(r[\'family\'] for r in rows)),\n        \'sameContextMultipleQuestions\':True,\'fieldAndQueryOrderIndependent\':True,\n        \'heldoutNames\':values[\'heldNames\'],\'heldoutColors\':values[\'heldColors\'],\n        \'practiceSampling\':\'80% new curriculum / 20% V19 replay within tutor updates; copying emphasis in first 200 updates\',\n        \'benchmarkPromptsExcluded\':True,\'teacherCalls\':0,\'holdoutScope\':\'Held out of V20 synthetic practice; historical public exposure is unknown.\'}\n\n\ndef grade_content(reply,expected,field=\'value\'):\n    """Conservative full-answer parsing, never keyword containment or a semantic judge."""\n    sensitive=field==\'code\'\n    if exact_grade(reply,expected,sensitive):return {\'verifiedContentCorrect\':True,\'strictFormatCorrect\':True,\'method\':\'exact\'}\n    text=reply.strip();parsed=None\n    patterns={\'name\':[r\'(?:Your|The) name is (.+?)[.!]?\'],\n      \'color\':[r\'(?:Your|The) scarf is (.+?)[.!]?\',r\'(?:Your|The) (?:scarf color|color) is (.+?)[.!]?\'],\n      \'code\':[r\'(?:Your|The) code is (.+?)[.!]?\'],\n      \'place\':[r\'(?:Your|The) (?:place|location) is (.+?)[.!]?\'],\n      \'number\':[r\'(?:Your|The) (?:number|sum) is (.+?)[.!]?\']}\n    for pattern in patterns.get(field,[])+[r\'The answer is (.+?)[.!]?\',r\'It is (.+?)[.!]?\']:\n        match=re.fullmatch(pattern,text,flags=re.I)\n        if match:parsed=match.group(1);break\n    if parsed is not None:\n        return {\'verifiedContentCorrect\':exact_grade(parsed,expected,sensitive),\'strictFormatCorrect\':False,\'method\':\'complete-sentence-pattern\'}\n    if re.fullmatch(r\'[A-Za-z0-9_-]+[.!]?\',text):\n        return {\'verifiedContentCorrect\':False,\'strictFormatCorrect\':False,\'method\':\'wrong-single-value\'}\n    return {\'verifiedContentCorrect\':None,\'strictFormatCorrect\':False,\'method\':\'pending-human-review\'}\n', 'v21_curriculum_data.py': '"""Progressive paired context practice; no teacher calls or user-chat imports."""\nimport random\nfrom collections import Counter\nfrom v20_generalize_data import (lessons as previous_lessons, inventory, profile, context,\n    QUESTIONS, held_math, pair, norm, key, window, collate, exact_grade, grade_content,\n    benchmarks)\n\n\ndef lessons(seed=210221,groups=1800):\n    rng=random.Random(seed);values=inventory(seed)\n    old,cases,_=previous_lessons(seed,groups)\n    banned={norm(m[\'content\']) for c in cases for m in c[\'messages\'] if m[\'role\']==\'user\'}\n    rows=[];seen=set()\n    def add(family,messages):\n        identifier=key(messages)\n        if identifier in seen or any(norm(m[\'content\']) in banned for m in messages if m[\'role\']==\'user\'):return\n        seen.add(identifier)\n        rows.append(dict(family=family,messages=messages,id=identifier,origin=\'original-synthetic-v21\'))\n    for row in old:\n        if row[\'family\'] in [\'updates\',\'instructions\']:continue\n        messages=row[\'messages\']\n        # Arithmetic gets sum supervision only: do not reward copying either operand in this family.\n        if row[\'family\']==\'arithmetic\':messages=messages[:3]\n        add(row[\'family\'],messages)\n    for i in range(groups):\n        p=profile(rng,values);fields=rng.sample([\'name\',\'color\',\'place\'],3)\n        for field in fields:\n            label={\'name\':\'Name\',\'color\':\'Scarf color\',\'place\':\'Location\'}[field]\n            add(\'single\',pair(f\'{label}: {p[field]}. \'+rng.choice(QUESTIONS[field])+\' Only the value.\',p[field]))\n        text=context(rng,p)\n        # Each question is trained alone AND after another, correct answer.\n        first,second=fields[:2]\n        for field in [first,second]:\n            add(\'contrast\',pair(text+\' \'+rng.choice(QUESTIONS[field])+\' Only the value.\',p[field]))\n        messages=pair(text+\' \'+rng.choice(QUESTIONS[first])+\' Only the value.\',p[first])\n        messages += [{\'role\':\'user\',\'content\':rng.choice(QUESTIONS[second])+\' Only the value.\'},\n                     {\'role\':\'assistant\',\'content\':p[second]}]\n        add(\'contrast\',messages)\n        # Rotate values and positions so first-item selection is not a useful shortcut.\n        items=[p[\'name\'],p[\'color\'],p[\'place\']];rng.shuffle(items)\n        for index in range(3):\n            q=rng.choice([f\'List: {" / ".join(items)}. Give item {index+1} only.\',\n                          f\'From {", ".join(items)}, return the \'+[\'first\',\'second\',\'third\'][index]+\' item only.\'])\n            add(\'instructions\',pair(q,items[index]))\n        new=rng.choice([c for c in values[\'colors\'] if c!=p[\'color\']])\n        # Identical history, independently supervised final queries; no "current color" assistant template.\n        history=pair(f\'My name is {p["name"]}. My scarf is {p["color"]}.\',\'Understood.\')\n        history += [{\'role\':\'user\',\'content\':f\'Correction: my scarf is {new} now.\'},\n                    {\'role\':\'assistant\',\'content\':\'Understood.\'}]\n        for field,value in [(\'name\',p[\'name\']),(\'color\',new)]:\n            add(\'updates\',history+[{\'role\':\'user\',\'content\':rng.choice(QUESTIONS[field])+\' Only the value.\'},\n                                   {\'role\':\'assistant\',\'content\':value}])\n        a=rng.randrange(31);b=rng.randrange(31)\n        if not held_math(a,b):\n            add(\'arithmetic\',pair(rng.choice([f\'Add {a} and {b}.\',f\'What is {a} + {b}?\',\n                f\'Compute the sum of {a} and {b}.\'])+\' Return a number only.\',str(a+b)))\n    # Ordinary requests include useful follow-ups, with varied topics and user-supplied facts.\n    topics=[(\'drawing\',\'Choose a simple object, sketch its outline, and add a few details.\'),\n            (\'gardening\',\'Choose a suitable plant, check its light needs, and keep the soil appropriately moist.\'),\n            (\'cooking\',\'Pick an easy recipe, read it first, and prepare the ingredients.\'),\n            (\'reading\',\'Choose a short book that interests you and set aside a little time each day.\'),\n            (\'writing\',\'Choose an idea, write a short draft, and revise it for clarity.\'),\n            (\'walking\',\'Choose a comfortable route and begin with a short walk.\'),\n            (\'organizing\',\'Choose one small area, sort the items, and put away what you want to keep.\'),\n            (\'studying\',\'Choose one topic, review the basics, and try a short practice exercise.\')]\n    for topic,advice in topics:\n        for ask in [f\'I want to start {topic}. Can you help?\',f\'How can I begin {topic}?\',\n                    f\'I am new to {topic}. Give me a simple first step.\']:\n            messages=pair(ask,advice)\n            messages += [{\'role\':\'user\',\'content\':\'I only have ten minutes today.\'},\n                {\'role\':\'assistant\',\'content\':f\'Start with one small {topic} task that fits into ten minutes. You can continue another day.\'}]\n            add(\'conversation\',messages)\n    for ask in [\'Hi!\',\'Hello! What can you help me with?\',\'What can you do?\', \'Can you help me?\']:\n        add(\'conversation\',pair(ask,\'I can help explain ideas, write and edit text, make plans, and practice questions. What would you like help with?\'))\n    for item in [\'a laptop\',\'a book\',\'a class\',\'a gift\',\'a game\',\'a recipe\',\'a place to visit\']:\n        add(\'conversation\',pair(f\'Help me choose {item}.\',\'What options are you considering, and what matters most to you?\'))\n    return rows,cases,dict(examples=len(rows),familyExamples=dict(Counter(r[\'family\'] for r in rows)),\n        teacherCalls=0,benchmarkPromptsExcluded=True,heldoutScope=\'V21 synthetic practice only; historical public exposure unknown.\',\n        curriculum=\'0-399: single facts; 400-999: contrasting queries; 1000+: people and updates; ordinary conversation throughout\',\n        practiceSampling=\'70% tutor updates / 30% raw updates; 15% V19 replay within tutor examples\',\n        arithmeticOperandFollowups=False)\n\n\ndef curriculum(step):\n    names=[\'copy\',\'single\',\'contrast\',\'binding\',\'entities\',\'updates\',\'instructions\',\'arithmetic\',\'meaning\',\'conversation\']\n    if step<400:weights=[15,25,10,5,2,2,10,6,5,20]\n    elif step<1000:weights=[8,10,22,10,8,8,10,6,5,13]\n    else:weights=[5,5,15,10,15,15,10,6,5,14]\n    return names,weights\n', 'v22_conversation_data.py': '"""Answer questions, acknowledge statements, and practice grounded short dialogues."""\nimport random\nfrom collections import Counter\nfrom v21_curriculum_data import lessons as previous_lessons\nfrom v20_generalize_data import (inventory, profile, benchmarks, QUESTIONS, pair, norm, key,\n    window, collate, exact_grade, grade_content, held_math)\n\n\ndef lessons(seed=220221,groups=1800):\n    rng=random.Random(seed);values=inventory(seed);cases=benchmarks(seed)\n    banned={norm(m[\'content\']) for c in cases for m in c[\'messages\'] if m[\'role\']==\'user\'}\n    rows=[];seen=set()\n    def add(family,messages,origin=\'original-synthetic-v22\'):\n        identifier=key(messages)\n        if identifier in seen or any(norm(m[\'content\']) in banned for m in messages if m[\'role\']==\'user\'):return\n        seen.add(identifier);rows.append(dict(family=family,messages=messages,id=identifier,origin=origin))\n    # Practice that V21 learned is a minority replay source, rather than the main curriculum.\n    old,_,_=previous_lessons(seed,groups)\n    for row in old:add(\'replay/\'+row[\'family\'],row[\'messages\'],\'historical-synthetic-replay\')\n    asks={\n        \'name\':[\'What is my name?\',\'Which name did I give you?\',\'Tell me my name.\'],\n        \'color\':[\'What color is my scarf?\',\'Which scarf color did I mention?\',\'Tell me the color of my scarf.\'],\n        \'place\':[\'Where is my scarf?\',\'Which location did I give for the scarf?\',\'Tell me where I put my scarf.\']}\n    topics=[\n        (\'drawing\',\'sketch a cup\',\'Draw the outline first, then add the handle and a few shadows.\'),\n        (\'writing\',\'draft a short paragraph\',\'Choose one idea, write a few sentences, and read them back.\'),\n        (\'reading\',\'read a short chapter\',\'Pick a book that interests you and read a few pages without rushing.\'),\n        (\'organizing\',\'clear one shelf\',\'Take the items off, decide what to keep, and put them back neatly.\'),\n        (\'studying\',\'practice one topic\',\'Review an example, try a similar question, and check your answer.\'),\n        (\'gardening\',\'check a plant\',\'Read its light and watering needs before choosing where to put it.\'),\n        (\'walking\',\'take a short walk\',\'Choose a comfortable route and begin at an easy pace.\'),\n        (\'cooking\',\'choose an easy recipe\',\'Read the recipe first, gather the ingredients, and follow the steps.\'),\n        (\'photography\',\'photograph an object\',\'Use good light, check the background, and try two different angles.\'),\n        (\'learning a language\',\'practice a few words\',\'Learn their meanings, say them aloud, and use them in short sentences.\'),\n        (\'planning\',\'write a short task list\',\'List what needs doing, choose a priority, and start with one task.\'),\n        (\'editing\',\'revise a paragraph\',\'Check the meaning first, then improve unclear wording and spelling.\')]\n    for i in range(groups):\n        p=profile(rng,values)\n        facts=f\'My name is {p["name"]}. My scarf is {p["color"]}. It is in the {p["place"]}.\'\n        # Identical facts with and without a query. Acknowledgment is correct ONLY for the statement.\n        add(\'direct\',pair(facts,\'Thanks for telling me.\'))\n        for field in [\'name\',\'color\',\'place\']:\n            q=rng.choice(asks[field]);answer=p[field]\n            add(\'direct\',pair(facts+\' \'+q,answer))\n            history=pair(facts,\'Thanks for telling me.\')\n            history += [{\'role\':\'user\',\'content\':q},{\'role\':\'assistant\',\'content\':answer}]\n            add(\'direct\',history)\n        other=profile(rng,values)\n        while other[\'name\']==p[\'name\'] or other[\'color\']==p[\'color\']:other=profile(rng,values)\n        people=[p,other];rng.shuffle(people)\n        record=\' \'.join(f\'{person["name"]} has a {person["color"]} scarf.\' for person in people)\n        for person in people:\n            q=rng.choice([f\'What color is {person["name"]}\\\'s scarf?\',\n                          f\'Which scarf color belongs to {person["name"]}?\',\n                          f\'Give the scarf color for {person["name"]}.\'])\n            add(\'ownership\',pair(record+\' \'+q,person[\'color\']))\n        dialogue=pair(record+f\' What color is {people[0]["name"]}\\\'s scarf?\',people[0][\'color\'])\n        dialogue += [{\'role\':\'user\',\'content\':f\'And what color is {people[1]["name"]}\\\'s scarf?\'},\n                     {\'role\':\'assistant\',\'content\':people[1][\'color\']}]\n        add(\'ownership\',dialogue)\n        topic,task,advice=topics[i%len(topics)]\n        minutes=rng.choice([10,15,20,30]);name=p[\'name\']\n        question=rng.choice([f\'I want to start {topic}. What should I do?\',\n            f\'Can you help me with {topic}? I am a beginner.\',f\'Give me an easy way to begin {topic}.\'])\n        answer=f\'You could {task}. {advice}\'\n        dialogue=pair(question,answer)\n        dialogue += [{\'role\':\'user\',\'content\':f\'I have {minutes} minutes today. Make a short plan.\'},\n            {\'role\':\'assistant\',\'content\':f\'Use your {minutes} minutes to {task}. Keep the task small and continue another day if needed.\'},\n            {\'role\':\'user\',\'content\':\'What topic are we working on?\'},\n            {\'role\':\'assistant\',\'content\':f\'We are working on {topic}.\'}]\n        add(\'dialogue\',dialogue)\n        favor=rng.choice([\'watering my plants\',\'lending me a book\',\'helping me move\',\'feeding my cat\',\'your advice\'])\n        dialogue=pair(f\'Write a short thank-you message to {name} for {favor}.\',\n                      f\'Hi {name}, thank you for {favor}. I really appreciate your help.\')\n        dialogue += [{\'role\':\'user\',\'content\':\'Make it shorter, but keep the name and reason.\'},\n            {\'role\':\'assistant\',\'content\':f\'Thank you for {favor}, {name}.\'},\n            {\'role\':\'user\',\'content\':\'Who is the message for?\'},{\'role\':\'assistant\',\'content\':name}]\n        add(\'dialogue\',dialogue)\n        event=rng.choice([\'a picnic\',\'a study session\',\'a birthday lunch\',\'a book club meeting\'])\n        day=rng.choice([\'Monday\',\'Tuesday\',\'Friday\',\'Saturday\']);place=p[\'place\']\n        people=rng.choice([2,3,4,6,8]);hour=rng.randrange(9,18)\n        start=f\'{hour}:00\';field=rng.choice([\'day\',\'time\',\'people\'])\n        followup={\'day\':(\'Which day did I choose?\',day),\'time\':(\'What time are we starting?\',start),\n                  \'people\':(\'How many people are coming?\',str(people))}[field]\n        dialogue=pair(f\'Help me plan {event}.\',\'When will it happen, where will it be, and how many people are coming?\')\n        dialogue += [{\'role\':\'user\',\'content\':f\'{day} at {start}, in the {place}, with {people} people.\'},\n            {\'role\':\'assistant\',\'content\':f\'For {event} on {day} at {start} in the {place}, prepare space for {people} people and a short list of what to bring.\'},\n            {\'role\':\'user\',\'content\':followup[0]},{\'role\':\'assistant\',\'content\':followup[1]}]\n        add(\'dialogue\',dialogue)\n        a=rng.randrange(31);b=rng.randrange(31)\n        if not held_math(a,b):add(\'arithmetic\',pair(f\'What is {a} + {b}? Give only the sum.\',str(a+b)))\n    return rows,cases,dict(examples=len(rows),familyExamples=dict(Counter(r[\'family\'] for r in rows)),\n        teacherCalls=0,benchmarkPromptsExcluded=True,\n        heldoutScope=\'V22 synthetic benchmark prompts/values excluded from V22 practice; historical public exposure unknown.\',\n        practiceSampling=\'70% tutor updates / 30% raw updates. Within tutor: 15% previous synthetic practice replay; 85% new lessons.\',\n        newFamilySampling=\'45% dialogue, 30% direct answers versus acknowledgments, 20% ownership, 5% arithmetic\',\n        naturalPublicConversationsAdded=False,\n        notice=\'Original template-generated multi-turn lessons; variety is limited and this is not proof of conversational ability.\')\n\n\ndef curriculum(step):\n    return [\'dialogue\',\'direct\',\'ownership\',\'arithmetic\'],[45,30,20,5]\n', 'v23_public_chat.py': '"""Pinned public conversations, complete within context; RAM to HF only."""\nimport fnmatch\nimport json\nimport random\nimport re\nfrom collections import Counter\nfrom nebula import encode\nfrom v18_stream import digest,emit,SECRET\nfrom v19_tutor_data import norm\n\nDATASET=\'HuggingFaceTB/smol-smoltalk\'\nREVISION=\'f73fe857d519ff6ac5af2ea67c4d3834da7b8bcc\'\nSPEC=dict(dataset=DATASET,revision=REVISION,license=\'apache-2.0\',url=\'https://huggingface.co/datasets/\'+DATASET,\n          terms=\'https://www.apache.org/licenses/LICENSE-2.0\',split=\'train\',synthetic=True)\n\n\ndef records(hub,cursor,seed):\n    from huggingface_hub import HfFileSystem\n    import pyarrow.parquet as pq\n    info=hub.api.dataset_info(DATASET,revision=REVISION)\n    card=info.card_data.to_dict() if hasattr(info.card_data,\'to_dict\') else dict(info.card_data or {})\n    patterns=[]\n    for item in card.get(\'configs\',[]):\n        if item.get(\'config_name\')==\'default\':\n            for entry in item.get(\'data_files\',[]):\n                if entry.get(\'split\')==\'train\':\n                    value=entry[\'path\'];patterns.extend(value if isinstance(value,list) else [value])\n    if not patterns:patterns=[\'data/train-*.parquet\',\'train/*.parquet\']\n    paths=sorted(p for p in hub.api.list_repo_files(DATASET,repo_type=\'dataset\',revision=REVISION)\n                 if p.endswith(\'.parquet\') and any(fnmatch.fnmatch(p,pattern) for pattern in patterns))\n    if not paths:raise ValueError(\'Pinned public dataset has no declared train parquet files\')\n    random.Random(seed).shuffle(paths);fs=HfFileSystem(token=hub.token)\n    for file_index in range(cursor[\'file\'],len(paths)):\n        start=cursor[\'row\'] if file_index==cursor[\'file\'] else 0\n        with fs.open(f\'datasets/{DATASET}@{REVISION}/{paths[file_index]}\',\'rb\',block_size=4*1024*1024,cache_type=\'bytes\') as stream:\n            parquet=pq.ParquetFile(stream)\n            # Parquet schema.names lists nested leaves; Arrow names preserve top-level messages.\n            if \'messages\' not in parquet.schema_arrow.names:raise ValueError(\'Public chat messages column missing\')\n            columns=[\'messages\']+([\'source\'] if \'source\' in parquet.schema_arrow.names else [])\n            offset=0\n            for group in range(parquet.num_row_groups):\n                size=parquet.metadata.row_group(group).num_rows\n                if offset+size<=start:offset+=size;continue\n                for batch in parquet.iter_batches(batch_size=32,row_groups=[group],columns=columns):\n                    for row in batch.to_pylist():\n                        index=offset;offset+=1\n                        if index<start:continue\n                        yield row,dict(file=file_index,row=index+1),paths[file_index]+\':\'+str(index)\n\n\ndef convert(item,context,banned):\n    messages=item.get(\'messages\')\n    if not isinstance(messages,list) or not 2<=len(messages)<=12:return None,\'structure\'\n    clean=[];expected=\'user\'\n    for i,message in enumerate(messages):\n        if not isinstance(message,dict):return None,\'structure\'\n        role=message.get(\'role\');text=message.get(\'content\')\n        if not isinstance(text,str) or not text.strip():return None,\'structure\'\n        text=text.replace(\'\\r\\n\',\'\\n\').replace(\'\\r\',\'\\n\').strip()\n        if role==\'system\' and i==0:pass\n        elif role==expected:expected=\'assistant\' if role==\'user\' else \'user\'\n        else:return None,\'structure\'\n        if SECRET.search(text) or re.search(r\'[\\w.+-]+@[\\w.-]+\\.[A-Za-z]{2,}\',text):return None,\'contact_or_credential\'\n        if \'\\ufffd\' in text or any(ord(c)<32 and c not in \'\\n\\t\' for c in text):return None,\'encoding\'\n        if role==\'user\' and norm(text) in banned:return None,\'benchmark\'\n        if role in [\'system\',\'assistant\'] and re.search(r\'\\b(ChatGPT|OpenAI|Claude|Gemini|Llama|HuggingChat)\\b\',text,re.I):return None,\'provider_identity\'\n        clean.append(dict(role=role,content=text))\n    if clean[-1][\'role\']!=\'assistant\':return None,\'incomplete\'\n    try:tokens,_=encode(clean)\n    except (ValueError,UnicodeError):return None,\'encoding\'\n    if len(tokens)>context+1:return None,\'too_long_no_truncation\'\n    return dict(family=\'public\',messages=clean,id=digest(clean),origin=\'public-synthetic\',\n                source={**SPEC,\'component\':str(item.get(\'source\',\'unknown\'))[:100],\n                        \'transformations\':[\'CRLF normalized; outer whitespace stripped; complete conversation retained\'],\n                        \'humanVerified\':False}),None\n\n\ndef prepare_public(hub,state,config,context,banned,reader=records):\n    prep=dict(state.get(\'publicPreparation\') or dict(cursor=dict(file=0,row=0),scanned=0,excluded={},complete=False))\n    rows=json.loads(hub.read_entry(state[\'publicRows\'])) if state.get(\'publicRows\') else []\n    seen={row[\'id\'] for row in rows};counts=Counter(prep[\'excluded\'])\n    target=config[\'publicConversations\'];limit=config[\'publicMaxScanned\']\n    def save():\n        nonlocal state\n        prep.update(excluded=dict(counts),accepted=len(rows))\n        state=hub.publish({**state,\'publicPreparation\':dict(prep)},\n                         {\'publicRows\':(\'public-conversations.json\',json.dumps(rows).encode())})\n        emit(v23PublicPreparationSaved=True,scanned=prep[\'scanned\'],accepted=len(rows),complete=prep[\'complete\'])\n    if prep[\'complete\']:return state,rows\n    if not state.get(\'publicSourceCard\'):\n        card=hub.fetch(DATASET,\'README.md\',REVISION,dataset=True)\n        state=hub.publish(state,{\'publicSourceCard\':(\'smol-smoltalk-README.md\',card),\n              \'publicSourceAttribution\':(\'public-attribution.json\',json.dumps(SPEC,indent=2).encode())})\n    try:\n        for item,cursor,locator in reader(hub,prep[\'cursor\'],config[\'seed\']):\n            if len(rows)>=target or prep[\'scanned\']>=limit:break\n            prep[\'cursor\']=cursor;prep[\'scanned\']+=1\n            row,reason=convert(item,context,banned)\n            if row and row[\'id\'] not in seen:\n                seen.add(row[\'id\']);row[\'source\'][\'locator\']=locator;rows.append(row)\n            else:counts[reason or \'duplicate\']+=1\n            if prep[\'scanned\']%2000==0:\n                emit(v23PublicProgress=True,scanned=prep[\'scanned\'],accepted=len(rows),target=target)\n            if len(rows) and len(rows)%500==0 and prep.get(\'accepted\')!=len(rows):save()\n        if len(rows)<min(target,256):\n            save();raise ValueError(\'Not enough complete public conversations fit the context; preparation saved. Inspect exclusions before training.\')\n        prep[\'complete\']=True;prep[\'targetReached\']=len(rows)>=target;save()\n    except BaseException:\n        if len(rows)!=prep.get(\'accepted\',0):save()\n        raise\n    return state,rows\n\n\ndef partition(rows):\n    # The whole conversation stays in one split, including every assistant turn.\n    train=[row for row in rows if int(row[\'id\'][:8],16)%20!=0]\n    valid=[row for row in rows if int(row[\'id\'][:8],16)%20==0]\n    if not train or not valid:raise ValueError(\'Public conversation holdout is empty; acquire more records\')\n    return train,valid\n', 'v24_context_data.py': '"""Original, varied context practice; fixed V23 benchmark values stay held out."""\nimport json\nimport random\nfrom collections import Counter\nfrom nebula import encode\nfrom v20_generalize_data import inventory, profile, pair, key, norm\nfrom v18_foundation_control import read_source\nfrom v23_public_chat import convert\n\n\ndef context_lessons(seed, cases, groups=2400, context=1024):\n    rng=random.Random(seed+240024);values=inventory(seed);rows=[];seen=set()\n    banned={norm(m[\'content\']) for c in cases for m in c[\'messages\'] if m[\'role\']==\'user\'}\n    def add(messages,kind):\n        identifier=key(messages)\n        if identifier in seen or len(encode(messages)[0])>context+1:return\n        if any(norm(m[\'content\']) in banned for m in messages if m[\'role\']==\'user\'):return\n        seen.add(identifier);rows.append(dict(id=identifier,family=\'context/\'+kind,messages=messages,\n            origin=\'original-synthetic-v24\',humanVerified=False))\n    objects=[\'cup\',\'bag\',\'coat\',\'scarf\',\'notebook\',\'bottle\',\'hat\',\'box\']\n    for i in range(groups):\n        a=profile(rng,values);b=profile(rng,values)\n        while a[\'name\']==b[\'name\'] or a[\'color\']==b[\'color\']:b=profile(rng,values)\n        obj=rng.choice(objects);people=[a,b];rng.shuffle(people)\n        facts=[f\'{p["name"]} has a {p["color"]} {obj} in the {p["place"]}.\' for p in people]\n        for p in people:\n            question=rng.choice([f\'What color is {p["name"]}\\\'s {obj}?\',\n                f\'Give the {obj} color belonging to {p["name"]}.\',f\'For {p["name"]}, which {obj} color was mentioned?\'])\n            add(pair(\' \'.join(facts)+\' \'+question+\' Give only the color.\',p[\'color\']),\'ownership\')\n            add(pair(\' \'.join(facts)+f\' Where is {p["name"]}\\\'s {obj}? Reply with only the place.\',p[\'place\']),\'location\')\n        first,second=people\n        dialogue=pair(\' \'.join(facts)+f\' Which color belongs to {first["name"]}?\',first[\'color\'])\n        dialogue+=pair(f\'And the {obj} owned by {second["name"]}? Give its color.\',second[\'color\'])[1:]\n        add(dialogue,\'multi-turn\')\n        updated=rng.choice([c for c in values[\'colors\'] if c not in [a[\'color\'],b[\'color\']]])\n        dialogue=pair(\' \'.join(facts),\'I understand the two descriptions.\')\n        correction=rng.choice([f\'{a["name"]}\\\'s {obj} is {updated} now. The other person\\\'s {obj} stays the same.\',\n            f\'Correction: change only {a["name"]}\\\'s {obj} color to {updated}.\'])\n        dialogue+=pair(correction,\'Thanks for the correction.\')[1:]\n        queries=[(a[\'name\'],updated),(b[\'name\'],b[\'color\'])];rng.shuffle(queries)\n        for name,color in queries:dialogue+=pair(f\'What is the current color of {name}\\\'s {obj}? Only the color.\',color)[1:]\n        add(dialogue,\'correction\')\n        fields=[(\'person\',a[\'name\']),(\'color\',a[\'color\']),(\'location\',a[\'place\']),(\'tag\',a[\'code\'])]\n        rng.shuffle(fields);record=\'; \'.join(f\'{k}: {v}\' for k,v in fields)\n        for field,value in rng.sample(fields,2):\n            add(pair(record+f\'. Read the {field} field. Return its value only.\',value),\'field-order\')\n        dialogue=pair(f\'My name is {a["name"]}. My {obj} is {a["color"]}.\',\'Thanks for telling me.\')\n        dialogue+=pair(f\'What color is my {obj}? Only the color.\',a[\'color\'])[1:]\n        dialogue+=pair(\'Which name did I provide? Only the name.\',a[\'name\'])[1:]\n        add(dialogue,\'query-switch\')\n        # Copy arbitrary training strings rather than choosing answers from a tiny list.\n        add(pair(f\'The label reads "{a["code"]}". Copy the label text without quotation marks.\',a[\'code\']),\'copy\')\n    return rows,dict(examples=len(rows),families=dict(Counter(r[\'family\'] for r in rows)),\n        templateGenerated=True,heldoutValuesExcluded=True,benchmarkPromptsExcluded=True,\n        notice=\'Synthetic context practice; success on these templates does not establish open conversation ability.\')\n\n\ndef prepare_public(hub,state,config,context,banned):\n    """Read the saved V23 public snapshot in RAM; never train from V23 weights."""\n    source,saved=read_source(config[\'sourceRepo\'],config[\'publicSourceRun\'],config[\'publicSourceRevision\'],hub.token)\n    if not saved.get(\'publicPreparation\',{}).get(\'complete\') or not saved.get(\'publicRows\'):\n        raise ValueError(\'Pinned V23 public preparation is not complete\')\n    original=json.loads(source.read_entry(saved[\'publicRows\']))\n    rows=[];excluded=Counter()\n    for row in original:\n        checked,reason=convert({\'messages\':row[\'messages\']},context,banned)\n        if reason:excluded[reason]+=1\n        else:rows.append(row)\n    if len(rows)<256:raise ValueError(\'Too few reusable public conversations\')\n    # Retain provenance and an immutable pointer, not duplicate data or weights.\n    reference=dict(repo=config[\'sourceRepo\'],run=config[\'publicSourceRun\'],revision=source.revision,\n        sha256=saved[\'publicRows\'][\'sha256\'],accepted=len(rows),excluded=dict(excluded))\n    if state.get(\'publicReference\')!=reference:\n        state=hub.publish({**state,\'publicReference\':reference})\n    return state,rows\n', 'v25_answer_data.py': '"""Target answer turns, not intermediate acknowledgments; preserve all context."""\nimport random\nfrom collections import Counter\nfrom nebula import encode,ASSISTANT\nfrom v24_context_data import context_lessons,prepare_public\nfrom v20_generalize_data import inventory,pair,key,norm\nfrom v19_tutor_data import window\n\n\ndef training_window(row,context):\n    x,y=window(row[\'messages\'],context)\n    if row.get(\'finalAnswerOnly\'):\n        if row[\'messages\'][-1][\'role\']!=\'assistant\':raise ValueError(\'A final assistant answer is required\')\n        tokens,_=encode(row[\'messages\']);last=max(i for i,t in enumerate(tokens) if t==ASSISTANT)\n        y[:last]=-100\n        if not (y!=-100).any():raise ValueError(\'No final answer targets\')\n    return x,y\n\n\ndef repair_lessons(seed,cases,groups=1800,context=1024):\n    base,_=context_lessons(seed,cases,groups,context);rows=[];seen=set()\n    banned={norm(m[\'content\']) for c in cases for m in c[\'messages\'] if m[\'role\']==\'user\'}\n    def add(messages,family):\n        if len(encode(messages)[0])>context+1 or any(norm(m[\'content\']) in banned for m in messages if m[\'role\']==\'user\'):return\n        identifier=key(messages)\n        if identifier in seen:return\n        seen.add(identifier);rows.append(dict(id=identifier,family=\'repair/\'+family,messages=messages,\n            finalAnswerOnly=True,origin=\'original-synthetic-v25\',humanVerified=False))\n    acknowledgments={\'I understand the two descriptions.\',\'Thanks for the correction.\',\'Thanks for telling me.\'}\n    # Each actual answer becomes its own complete context -> final-answer training example.\n    # Earlier assistant text remains visible but receives no loss for these examples.\n    for row in base:\n        family=row[\'family\'].split(\'/\',1)[1]\n        for i,m in enumerate(row[\'messages\']):\n            if m[\'role\']==\'assistant\' and m[\'content\'] not in acknowledgments:\n                add(row[\'messages\'][:i+1],family)\n    rng=random.Random(seed+250025);values=inventory(seed)\n    topics=[(\'drawing\',\'sketch a simple object\',\'draw its outline, then add one detail\'),\n        (\'writing\',\'draft a few sentences\',\'choose one idea and write about it\'),\n        (\'reading\',\'read a few pages\',\'choose a short passage and read it slowly\'),\n        (\'organizing\',\'clear a small space\',\'choose one shelf and sort its contents\'),\n        (\'studying\',\'practice one example\',\'review a worked example, then try one yourself\'),\n        (\'cooking\',\'prepare a simple dish\',\'choose a recipe and gather the ingredients\'),\n        (\'gardening\',\'check one plant\',\'check its soil and light before watering\'),\n        (\'photography\',\'photograph a nearby object\',\'use clear light and try two angles\')]\n    favors=[(\'watered my plants\',\'watering my plants\'),(\'lent me a book\',\'lending me a book\'),\n        (\'helped me move\',\'helping me move\'),(\'fed my cat\',\'feeding my cat\'),(\'shared their notes\',\'sharing your notes\')]\n    for i in range(groups):\n        name=rng.choice(values[\'names\']);topic,task,advice=topics[i%len(topics)];minutes=rng.randrange(5,46)\n        request=rng.choice([f\'I am a beginner at {topic}. Can you suggest a small task?\',f\'Help me start {topic} with something simple.\',f\'I want to practice {topic}. Please help me begin.\'])\n        messages=pair(request,\'How much time do you have?\')\n        question=rng.choice([f\'I have {minutes} minutes. Give me a simple first step.\',f\'There are {minutes} minutes available. What should I try first?\',f\'Just {minutes} minutes today. Suggest a short activity.\'])\n        answer=rng.choice([f\'Use your {minutes} minutes to {task}. First, {advice}.\',f\'Try to {task} during your {minutes} minutes. Start by taking this step: {advice}.\'])\n        messages+=pair(question,answer)[1:]\n        add(messages,\'practical\')\n        messages+=pair(\'Which activity did I ask about? Give only the activity.\',topic)[1:];add(messages,\'practical\')\n        past,thanks=rng.choice(favors)\n        messages=pair(\'Help me write a friendly thank-you message.\',\'Who is it for, and what did they do?\')\n        messages+=pair(f\'The message is for {name}. They {past}.\',f\'Hi {name}, thank you for {thanks}. I appreciate your help.\')[1:]\n        add(messages,\'writing\')\n        messages+=pair(\'Keep only the name and the reason in a shorter message.\',f\'{name}, thank you for {thanks}.\')[1:];add(messages,\'writing\')\n    return rows,dict(examples=len(rows),families=dict(Counter(r[\'family\'] for r in rows)),\n        objective=\'Final assistant answer only on repair records; full preceding context retained\',\n        intermediateAcknowledgmentTargets=0,templateGenerated=True,benchmarkPromptsExcluded=True,\n        notice=\'Synthetic repair curriculum. Existing 96 questions are a development comparison, not an independent certification of intelligence.\')\n', 'v25_answers.py': '"""Bounded tutor SFT plus general-language replay, with protected V18 control."""\nimport copy\nimport gc\nimport json\nimport math\nimport os\nimport random\nimport re\nimport time\nfrom collections import Counter\nfrom pathlib import Path\n\nimport numpy as np\nimport requests\nimport torch\nimport torch.nn.functional as F\nfrom v18_stream import Hub,RemotePool,deserialize,serialize,sha,digest,emit,PROBES\nfrom v18_foundation_control import read_source,heldout,SOURCE_REVISION\nfrom v18_decode_probe import generate,compare\nfrom v22_conversation_data import lessons,window,collate,exact_grade,grade_content,curriculum\nfrom v23_public_chat import partition\nfrom v25_answer_data import prepare_public,repair_lessons as context_lessons,training_window\nfrom v19_tutor_data import norm as normalize_prompt\n\nDEFAULT={\'sourceRepo\':\'iiidevv/nebula-raw-training\',\'sourceRun\':\'nebula-v22-conversation\',\n \'sourceRevision\':\'6c670c3670444d7cde2b17760fea1cdde4a93b18\',\'expectedSourceUpdates\':4000,\n \'rawCorpusRun\':\'nebula-v18-raw-1gb\',\'rawCorpusRevision\':SOURCE_REVISION,\'updates\':1000,\'seed\':230221,\n \'learningRate\':.000003,\'warmup\':60,\'batch\':8,\'microbatch\':1,\'saveEvery\':100,\'evaluateEvery\':500,\'publicSourceRun\':\'nebula-v23-longrun\',\'publicSourceRevision\':\'c21aafa220d33b9787791fcaf7aebf413bc62cf8\',\n \'maxSessionHours\':7.5}\nMOJIBAKE=(\'\\ufffd\',\'\\u00e2\\u20ac\',\'\\u00c3\\u00a2\',\'\\u00ef\\u00bf\\u00bd\',\'\\u00f0\\u0178\')\nOPEN=[\'Hello! What can you help me with?\',\'Explain why ice melts.\',\n      \'Write a thank-you note to a neighbor.\',\'Which should I choose?\',\n      \'My name is Iris. My scarf is purple. What color is my scarf?\',\n      \'I want to learn something new. Can you help me get started?\',\n      \'I have ten minutes to learn drawing. Suggest a first step.\',\n      "Nora has a green cup. Omar has a white cup. What color is Omar\'s cup?",\n      \'Write a friendly message thanking Luis for fixing my bicycle.\']\n\n\nclass TutorStore(Hub):\n    def __init__(self,repo,token,run):\n        from huggingface_hub import HfApi\n        if not re.fullmatch(r\'[\\w-]+/[\\w.-]+\',repo) or not re.fullmatch(r\'nebula-v25-[\\w-]{1,45}\',run):\n            raise ValueError(\'Use a separate nebula-v25 output run\')\n        self.repo=repo;self.token=token;self.prefix=\'experiments/\'+run+\'/\'\n        self.api=HfApi(token=token);self.session=requests.Session()\n        if self.api.whoami()[\'name\']!=repo.split(\'/\')[0]:raise ValueError(\'Use your own output namespace\')\n        info=self.api.model_info(repo)\n        if info.private is not False:raise ValueError(\'The configured output repository must be public\')\n        self.revision=info.sha\n\n\ndef signature(config):\n    root=Path(__file__).parent\n    code=\'\'.join((root/name).read_text(encoding=\'utf-8\') for name in\n                 [\'v25_answers.py\',\'v22_conversation_data.py\',\'v23_public_chat.py\',\'v24_context_data.py\',\'v25_answer_data.py\',\'v21_curriculum_data.py\',\'v20_generalize_data.py\',\'v19_tutor_data.py\',\'nebula.py\',\'v18_stream.py\',\'v18_decode_probe.py\',\'v18_foundation_control.py\'])\n    return digest({\'config\':config,\'codeSha256\':sha(code.encode())})\n\n\ndef preservation_rows(source,source_state,fallback,cases):\n    rows=fallback\n    if source_state.get(\'practice\'):\n        rows=[json.loads(line) for line in source.read_entry(source_state[\'practice\']).decode(\'utf-8\').splitlines() if line.strip()]\n    banned={normalize_prompt(m[\'content\']) for c in cases for m in c[\'messages\'] if m[\'role\']==\'user\'}\n    return [r for r in rows if not any(normalize_prompt(m[\'content\']) in banned for m in r[\'messages\'] if m[\'role\']==\'user\')]\n\n\ndef candidate_state(state,report,baseline):\n    protected=[\'entity-binding\',\'updated-facts\',\'unseen-values\',\'unseen-wording\',\'verbatim-copy\']\n    axes=all(report[\'byAxis\'].get(k,{}).get(\'correct\',0)>=baseline[\'byAxis\'].get(k,{}).get(\'correct\',0) for k in protected)\n    qualifies=(axes and report[\'exactCorrect\']>=baseline[\'exactCorrect\']\n        and report[\'heldoutMixed\'][\'loss\']<=baseline[\'heldoutMixed\'][\'loss\']*1.05\n        and report[\'heldoutPublic\'][\'loss\']<=baseline[\'heldoutPublic\'][\'loss\']*1.05)\n    if qualifies and report[\'exactCorrect\']>state.get(\'bestCandidateCorrect\',baseline[\'exactCorrect\']):\n        return {**state,\'bestCandidateCorrect\':report[\'exactCorrect\'],\'bestCandidate\':copy.deepcopy(state[\'working\']),\n            \'bestCandidateUpdates\':state[\'updates\'],\'candidateNotice\':\'Context benchmark improved without protected-axis regression or >5% loss regression; open replies still require inspection.\'}\n    return state\n\n\ndef raw_clean(pool,step,batch,seed):\n    result=[];rejected=0\n    for attempt in range(32):\n        for x,y in pool.batch(step,batch-len(result),seed+attempt*100003):\n            # Inspect text without changing the actual packed token stream. A window may cut a UTF-8 character.\n            text=bytes(int(v) for v in x if v<256).decode(\'utf-8\',errors=\'ignore\')\n            if any(pattern in text for pattern in MOJIBAKE):rejected+=1;continue\n            result.append((x,y))\n        if len(result)==batch:return result,rejected\n    raise ValueError(\'Could not sample a clean raw batch; last confirmed checkpoint remains saved\')\n\n\ndef tutor_batch(rows,windows,step,batch,seed):\n    families={}\n    for i,row in enumerate(rows):families.setdefault(row[\'family\'],[]).append(i)\n    rng=random.Random(seed+step*7919);pairs=[];counts=Counter()\n    context=sorted(k for k in families if k.startswith(\'repair/\'))\n    replay=sorted(k for k in families if k!=\'public\' and not k.startswith(\'repair/\'))\n    if not context or not replay or \'public\' not in families:raise ValueError(\'Context, replay and public practice are required\')\n    for _ in range(batch):\n        draw=rng.random()\n        family=rng.choices(context,weights=[{\'correction\':4,\'query-switch\':3,\'ownership\':2,\'multi-turn\':2,\'copy\':2,\'field-order\':1,\'location\':1,\'practical\':3,\'writing\':3}.get(k.split(\'/\',1)[1],1) for k in context],k=1)[0] if draw<.6 else (rng.choice(replay) if draw<.8 else \'public\')\n        index=rng.choice(families[family]);pairs.append(windows[index]);counts[family]+=1\n    return pairs,dict(counts)\n\n\n@torch.inference_mode()\ndef tutor_loss(model,windows):\n    total=0.;targets=0;device=next(model.parameters()).device;model.eval()\n    for pair in windows:\n        x,y=collate([pair]);count=int((y!=-100).sum())\n        loss=F.cross_entropy(model(x.to(device)).float().flatten(0,1),y.to(device).flatten(),reduction=\'sum\')\n        if not torch.isfinite(loss):raise ValueError(\'Non-finite held-out tutor loss\')\n        total+=float(loss);targets+=count\n    return {\'loss\':total/targets,\'assistantTargets\':targets}\n\n\ndef evaluate(model,cases,mixed_pairs,label,updates,full=True):\n    emit(v25EvaluationStarted=label,additionalUpdates=updates)\n    validations=[window(c[\'messages\']+[{\'role\':\'assistant\',\'content\':c[\'expected\']}],model.config.context) for c in cases]\n    losses={\'heldoutMixed\':heldout(model,mixed_pairs),\'heldoutTutor\':tutor_loss(model,validations)}\n    answers=[];axes={}\n    for i,case in enumerate(cases):\n        # Greedy, strict full-answer checks. Correct words inside an unrelated paragraph do not pass.\n        prompt=case[\'messages\'][-1][\'content\']\n        if len(case[\'messages\'])==2:\n            answer=generate(model,prompt,\'chat\',0,1,0,48)\n        else:\n            # Nebula.reply preserves the full multi-turn history and uses the same special-token convention.\n            text=model.reply(case[\'messages\'],max_tokens=48,temperature=0)\n            answer={\'reply\':text,\'stopReason\':\'see-generated-text\'}\n        correct=exact_grade(answer[\'reply\'],case[\'expected\'],case.get(\'field\')==\'code\')\n        content=grade_content(answer[\'reply\'],case[\'expected\'],case.get(\'field\',\'value\'))\n        axis=axes.setdefault(case[\'axis\'],{\'correct\':0,\'cases\':0,\'contentVerifiedCorrect\':0,\'contentPendingReview\':0})\n        axis[\'cases\']+=1;axis[\'correct\']+=int(correct)\n        axis[\'contentVerifiedCorrect\']+=int(content[\'verifiedContentCorrect\'] is True)\n        axis[\'contentPendingReview\']+=int(content[\'verifiedContentCorrect\'] is None)\n        answers.append({**case,**answer,\'exactCorrect\':correct,\'contentCheck\':content})\n        if (i+1)%8==0:emit(v25ExactProgress=i+1,total=len(cases))\n    report={\'stage\':label,\'additionalUpdates\':updates,**losses,\'exactCorrect\':sum(a[\'exactCorrect\'] for a in answers),\n            \'exactCases\':len(cases),\'byAxis\':axes,\'exactAnswers\':answers,\n            \'contentVerifiedCorrect\':sum(a[\'contentCheck\'][\'verifiedContentCorrect\'] is True for a in answers),\n            \'contentVerifiedIncorrect\':sum(a[\'contentCheck\'][\'verifiedContentCorrect\'] is False for a in answers),\n            \'contentPendingReview\':sum(a[\'contentCheck\'][\'verifiedContentCorrect\'] is None for a in answers),\n            \'contentRubric\':\'Only exact values or explicitly recognized complete sentence forms; arbitrary prose stays pending.\',\n            \'qualityApproved\':False,\'humanVerified\':False}\n    if full:\n        report[\'generation\']=compare(model,128,modes=[dict(name=\'greedy\',temperature=0.,top_p=1.,seeds=[0]),\n            dict(name=\'sample-0.7\',temperature=.7,top_p=.9,seeds=[17])],raw=PROBES,questions=OPEN)\n    emit(v25EvaluationSummary={k:v for k,v in report.items() if k not in [\'exactAnswers\',\'generation\']})\n    return report\n\n\ndef run_tutor(output_run=\'nebula-v25-answers\',config=None,device=\'cuda\',clock=None):\n    clock=time.monotonic if clock is None else clock;config=copy.deepcopy(DEFAULT if config is None else config)\n    if set(config)!=set(DEFAULT) or not 1<=config[\'updates\']<=100000 or not 0<config[\'maxSessionHours\']<=8:\n        raise ValueError(\'Invalid tutor experiment settings\')\n    for name in [\'batch\',\'microbatch\',\'saveEvery\',\'warmup\',\'evaluateEvery\']:\n        if type(config[name]) is not int or config[name]<1:raise ValueError(\'Invalid \'+name)\n    if config[\'microbatch\']>config[\'batch\'] or not 0<config[\'learningRate\']<=.001:raise ValueError(\'Invalid batch/learning rate\')\n    if device==\'cuda\' and not torch.cuda.is_available():raise RuntimeError(\'Enable a Kaggle NVIDIA GPU\')\n    token=os.environ.get(\'NEBULA_HF_TOKEN\') or os.environ.get(\'HF_TOKEN\')\n    if not token:raise RuntimeError(\'Enable Kaggle secret NEBULA_HF_TOKEN\')\n    source,source_state=read_source(config[\'sourceRepo\'],config[\'sourceRun\'],config[\'sourceRevision\'],token)\n    if source_state[\'updates\']!=config[\'expectedSourceUpdates\']:raise ValueError(\'Pinned source checkpoint step mismatch\')\n    corpus_source,corpus_state=read_source(config[\'sourceRepo\'],config[\'rawCorpusRun\'],config[\'rawCorpusRevision\'],token)\n    output=TutorStore(config[\'sourceRepo\'],token,output_run);state=output.state();run_signature=signature(config)\n    if state and state.get(\'signature\')!=run_signature:raise ValueError(\'Use identical notebook/settings to resume, or choose a NEW output run\')\n    if state and state.get(\'complete\'):\n        emit(v25AlreadyComplete=True,additionalUpdates=state[\'updates\'])\n        return json.loads(output.read_entry(state[\'finalReport\']))\n    rows,cases,data_report=lessons(config[\'seed\'])\n    rows=preservation_rows(source,source_state,rows,cases)\n    preservation_count=len(rows)\n    extra,context_report=context_lessons(config[\'seed\'],cases)\n    rows+=extra\n    data_report={**data_report,\'practiceSampling\':\'80% tutor updates / 20% raw updates; tutor examples 60% answer-turn repair / 20% V22 replay / 20% public conversations\',\n                 \'newFamilySampling\':\'Repair family weights: correction 4, query-switch/practical/writing 3, ownership/multi-turn/copy 2, field-order/location 1. Replay families uniform.\',\n                 \'publicSource\':\'HuggingFaceTB/smol-smoltalk\',\'publicPreparationRequired\':True,\n                 \'preservationSource\':\'saved V22 practice\' if source_state.get(\'practice\') else \'regenerated synthetic fallback\',\n                 \'preservationExamples\':preservation_count,\'answerRepairPractice\':context_report,\'familyExamples\':dict(Counter(r[\'family\'] for r in rows)),\n                 \'sourcePracticeSha256\':source_state.get(\'practice\',{}).get(\'sha256\'),\n                 \'publicSnapshotReused\':config[\'publicSourceRun\'],\n                 \'notice\':\'Preserved synthetic exercises plus new public synthetic conversations. Actual acquisition counts are logged at v25PublicReady; not human verified.\'}\n    data_report.pop(\'naturalPublicConversationsAdded\',None)\n    data_report[\'examples\']=len(rows)\n    data_report[\'repairObjective\']=\'Only final assistant answer bytes/EOS are targets for repair records; intermediate acknowledgments are context only.\'\n    if state is None:\n        state={\'format\':\'nebula-v25-answers-1\',\'signature\':run_signature,\'config\':config,\'updates\':0,\n             \'sourceRevision\':source.revision,\'sourceWeightSha256\':source_state[\'working\'][\'sha256\'],\n             \'sourceUpdates\':source_state[\'updates\'],\'complete\':False,\'qualityApproved\':False,\'publicWebsiteUpdated\':False,\n             \'uniqueTextBytes\':corpus_state[\'uniqueTextBytes\'],\'corpusComplete\':True,\n             \'rawCorpusRevision\':corpus_source.revision,\'rawCorpusRun\':config[\'rawCorpusRun\'],\n             \'corpusNotice\':\'References the existing pinned V18 corpus; the original data is not duplicated.\',\n             \'counters\':{\'tutorUpdates\':0,\'rawUpdates\':0,\'assistantTargets\':0,\'rawTargets\':0,\'filteredRawWindows\':0,\'familyExamples\':{}}}\n        state=output.publish(state,{\n            \'practice\':(\'tutor-practice.jsonl\',b\'\\n\'.join(json.dumps(row).encode() for row in rows)),\n            \'benchmark\':(\'heldout-benchmark.json\',json.dumps(cases,indent=2).encode()),\n            \'dataReport\':(\'data-report.json\',json.dumps(data_report,indent=2).encode())})\n    emit(v25Data=data_report,practiceMix={\'tutorUpdatesPercent\':80,\'rawUpdatesPercent\':20},\n         sourceCheckpoint=source_state[\'updates\'],sourceRun=config[\'sourceRun\'],\n         evaluationPolicy=\'baseline-then-every-500-updates-and-final\',sourceIsStoryControl=False,sourceModelResized=False,publicWebsiteUpdated=False)\n    model,payload=deserialize(output.read_entry(state[\'working\']) if state.get(\'working\') else source.read_entry(source_state[\'working\']))\n    banned={normalize_prompt(m[\'content\']) for c in cases for m in c[\'messages\'] if m[\'role\']==\'user\'}|{normalize_prompt(q) for q in OPEN}\n    state,public_rows=prepare_public(output,state,config,model.config.context,banned)\n    public_train,public_valid=partition(public_rows)\n    all_public_valid=public_valid\n    public_valid=public_valid[:64]\n    # Suppress exact duplicates of public validation conversations in synthetic replay.\n    valid_ids={r[\'id\'] for r in all_public_valid}\n    rows=[r for r in rows if r[\'id\'] not in valid_ids]+public_train\n    windows=[training_window(r,model.config.context) for r in rows]\n    public_windows=[window(r[\'messages\'],model.config.context) for r in public_valid]\n    emit(v25PublicReady=True,trainingConversations=len(public_train),validationConversations=len(public_valid),\n         tutorSampling=\'60% answer-turn repair / 20% V22 replay / 20% public conversations\',truncation=False)\n    model.to(device);torch.set_num_threads(2)\n    amp=device==\'cuda\';bf16=amp and torch.cuda.get_device_capability(0)[0]>=8 and torch.cuda.is_bf16_supported()\n    dtype=torch.bfloat16 if bf16 else torch.float16\n    scaler=torch.amp.GradScaler(\'cuda\',enabled=amp and not bf16,init_scale=1024)\n    optimizer=torch.optim.AdamW(model.parameters(),lr=config[\'learningRate\'],weight_decay=.01)\n    if state.get(\'working\'):\n        training=payload[\'trainingState\']\n        if training[\'signature\']!=run_signature or training[\'updates\']!=state[\'updates\']:raise ValueError(\'Tutor checkpoint mismatch\')\n        optimizer.load_state_dict(training[\'optimizer\'])\n        if scaler.is_enabled() and training.get(\'scaler\'):scaler.load_state_dict(training[\'scaler\'])\n        del training\n    del payload;gc.collect()\n    raw=RemotePool(corpus_source,corpus_state[\'corpus\'][\'train\'],model.config.context,64)\n    valid=RemotePool(corpus_source,corpus_state[\'corpus\'][\'validation\'],model.config.context,1)\n    mixed_pairs=[valid.batch(i,1,880019)[0] for i in range(16)]\n    del valid;gc.collect()\n    emit(v25TrainingSetup=True,parameters=sum(p.numel() for p in model.parameters()),trainingGPUs=1,\n         precision=str(dtype) if amp else \'float32\',optimizerRestored=bool(state.get(\'working\')),\n         optimizerResetReason=None if state.get(\'working\') else \'Changed generalization curriculum; V22 weights reused with a lower peak learning rate.\',\n         originalWeightsPreserved=True)\n    if not state.get(\'baselineReport\'):\n        baseline=evaluate(model,cases,mixed_pairs,\'baseline-v22-final\',0)\n        baseline[\'heldoutPublic\']=tutor_loss(model,public_windows)\n        emit(v25HeldoutPublic=baseline[\'heldoutPublic\'],stage=\'baseline\')\n        state=output.publish(state,{\'baselineReport\':(\'baseline.json\',json.dumps(baseline,indent=2).encode())})\n    baseline=json.loads(output.read_entry(state[\'baselineReport\']))\n    started=clock();last_saved=state[\'updates\'];bad=0;interrupted=False\n    def save():\n        nonlocal state,last_saved\n        stream=serialize(model,{\'experimental\':True,\'qualityApproved\':False,\'sourceUpdates\':source_state[\'updates\'],\n            \'additionalUpdates\':state[\'updates\']},{\'signature\':run_signature,\'updates\':state[\'updates\'],\n            \'optimizer\':optimizer.state_dict(),\'scaler\':scaler.state_dict()})\n        try:state=output.publish(state,{\'working\':(\'working.pt\',stream.getbuffer())})\n        finally:stream.close()\n        last_saved=state[\'updates\'];emit(v25CheckpointConfirmed=True,additionalUpdates=last_saved)\n    try:\n        while state[\'updates\']<config[\'updates\'] and clock()-started<config[\'maxSessionHours\']*3600:\n            step=state[\'updates\'];is_tutor=step%10<8\n            if is_tutor:pairs,families=tutor_batch(rows,windows,step,config[\'batch\'],config[\'seed\']);rejected=0\n            else:pairs,rejected=raw_clean(raw,step,config[\'batch\'],config[\'seed\']);families={}\n            rate=config[\'learningRate\']*min(1.,(step+1)/config[\'warmup\'])*(.2+.8*.5*(1+math.cos(math.pi*step/config[\'updates\'])))\n            for group in optimizer.param_groups:group[\'lr\']=rate\n            denominator=sum(int((y!=-100).sum()) for _,y in pairs)\n            model.train();optimizer.zero_grad(set_to_none=True);loss_value=0.\n            for offset in range(0,len(pairs),config[\'microbatch\']):\n                x,y=collate(pairs[offset:offset+config[\'microbatch\']]);x=x.to(device);y=y.to(device)\n                with torch.autocast(\'cuda\',dtype=dtype,enabled=amp):\n                    loss=F.cross_entropy(model(x).flatten(0,1),y.flatten(),reduction=\'sum\')/denominator\n                if not torch.isfinite(loss):raise ValueError(\'Non-finite loss; previous confirmed checkpoint preserved\')\n                loss_value+=float(loss.detach());scaler.scale(loss).backward()\n            scaler.unscale_(optimizer);norm=torch.nn.utils.clip_grad_norm_(model.parameters(),1.)\n            if not torch.isfinite(norm):\n                bad+=1\n                if not scaler.is_enabled() or bad>12:raise ValueError(\'Non-finite gradients\')\n                scaler.update(new_scale=scaler.get_scale()/2);continue\n            scaler.step(optimizer);scaler.update();bad=0;state[\'updates\']+=1\n            counters=state[\'counters\'];counters[\'tutorUpdates\' if is_tutor else \'rawUpdates\']+=1\n            counters[\'assistantTargets\' if is_tutor else \'rawTargets\']+=denominator;counters[\'filteredRawWindows\']+=rejected\n            for family,n in families.items():counters[\'familyExamples\'][family]=counters[\'familyExamples\'].get(family,0)+n\n            if state[\'updates\']%25==0:\n                emit(v25Update=state[\'updates\'],budget=config[\'updates\'],objective=\'assistant-only\' if is_tutor else \'raw-language\',\n                     loss=round(loss_value,4),learningRate=rate,counters=counters)\n            if state[\'updates\']%config[\'saveEvery\']==0:save()\n            if state[\'updates\']%config[\'evaluateEvery\']==0 and state[\'updates\']<config[\'updates\']:\n                if state[\'updates\']!=last_saved:save()\n                report=evaluate(model,cases,mixed_pairs,\'sparse-update-\'+str(state[\'updates\']),state[\'updates\'],full=False)\n                report[\'heldoutPublic\']=tutor_loss(model,public_windows)\n                emit(v25HeldoutPublic=report[\'heldoutPublic\'],stage=report[\'stage\'])\n                # Reference a context-improving candidate; never roll back or automatically release.\n                state=candidate_state(state,report,baseline)\n                state=output.publish(state,{\'latestEvaluation\':(\'sparse-evaluation.json\',json.dumps(report,indent=2).encode())})\n    except KeyboardInterrupt:interrupted=True\n    if state[\'updates\']!=last_saved:save()\n    if interrupted or state[\'updates\']<config[\'updates\']:\n        emit(v25Paused=True,updatesSaved=last_saved,runAgainWithSameSettings=True)\n        return state\n    final=evaluate(model,cases,mixed_pairs,\'tutor-final\',state[\'updates\'])\n    final[\'heldoutPublic\']=tutor_loss(model,public_windows)\n    emit(v25HeldoutPublic=final[\'heldoutPublic\'],stage=\'final\')\n    state=candidate_state(state,final,baseline)\n    report={\'format\':\'nebula-v25-answers-comparison-1\',\'baseline\':baseline,\'final\':final,\'counters\':state[\'counters\'],\n            \'sourceRevision\':source.revision,\'sourceWeightSha256\':source_state[\'working\'][\'sha256\'],\n            \'bestCandidateUpdates\':state.get(\'bestCandidateUpdates\'),\'bestCandidate\':state.get(\'bestCandidate\'),\n            \'mixedLossRatioToBaseline\':final[\'heldoutMixed\'][\'loss\']/baseline[\'heldoutMixed\'][\'loss\'],\n            \'qualityApproved\':False,\'humanVerified\':False,\'publicWebsiteUpdated\':False,\n            \'interpretation\':\'Compare development benchmark and real replies with V22. This benchmark has informed earlier curricula; it is not an independent final test. No automatic promotion or broad intelligence claim.\'}\n    report[\'mixedLossRegressionFlag\']=report[\'mixedLossRatioToBaseline\']>1.05\n    report[\'protectedAxisRegressions\']=[k for k in [\'entity-binding\',\'updated-facts\',\'unseen-values\',\'unseen-wording\',\'verbatim-copy\'] if final[\'byAxis\'].get(k,{}).get(\'correct\',0)<baseline[\'byAxis\'].get(k,{}).get(\'correct\',0)]\n    state=output.publish({**state,\'complete\':True},{\'finalReport\':(\'comparison.json\',json.dumps(report,indent=2).encode())})\n    emit(v25Finished=True,additionalUpdates=state[\'updates\'],exactCorrect=final[\'exactCorrect\'],exactCases=final[\'exactCases\'],\n         mixedLossRegressionFlag=report[\'mixedLossRegressionFlag\'],originalWeightsPreserved=True,publicWebsiteUpdated=False,\n         reportParts=[p[\'path\'] for p in state[\'finalReport\'][\'parts\']])\n    del model,optimizer;gc.collect()\n    if amp:torch.cuda.empty_cache()\n    return report\n'}
import importlib
code_root=Path.cwd()/'nebula-v25-answers-code'
code_root.mkdir(exist_ok=True)
for name,source in sources.items():(code_root/name).write_text(source,encoding='utf-8')
sys.path.insert(0,str(code_root))
for name in sources:sys.modules.pop(Path(name).stem,None)
importlib.invalidate_caches()
from v25_answers import run_tutor
comparison=run_tutor(OUTPUT_RUN,CONFIG)
if 'baseline' in comparison and 'final' in comparison:
    from IPython.display import display,Markdown
    import pandas as pd
    display(pd.DataFrame([{'stage':stage,'mixedLoss':comparison[stage]['heldoutMixed']['loss'],
          'tutorLoss':comparison[stage]['heldoutTutor']['loss'],'publicChatLoss':comparison[stage]['heldoutPublic']['loss'],
          'strictCorrect':comparison[stage]['exactCorrect'],'cases':comparison[stage]['exactCases'],
          'contentVerifiedCorrect':comparison[stage]['contentVerifiedCorrect'],
          'contentPendingReview':comparison[stage]['contentPendingReview']}
          for stage in ['baseline','final']]))
    for stage in ['baseline','final']:
        display(Markdown('## '+stage.title()))
        display(pd.DataFrame.from_dict(comparison[stage]['byAxis'],orient='index'))
        for row in comparison[stage]['exactAnswers']:
            print(row['axis'],'EXPECTED:',row['expected'],'REPLY:',row['reply'],
                  'STRICT:',row['exactCorrect'],'CONTENT:',row['contentCheck'])
        for row in comparison[stage].get('generation',{}).get('answers',[]):
            print(row['kind'],row['mode'],'PROMPT:',row['prompt'])
            print('REPLY:',row['reply'])
    print({'mixedLossRegressionFlag':comparison['mixedLossRegressionFlag'],'publicWebsiteUpdated':False})
else:
    print('Session paused. Run all again with identical settings to continue from the confirmed V25 checkpoint.')
